In [29]:
# Kaggle bootstrap — place this as the FIRST cell in the notebook
from pathlib import Path
import os, json

# 1) Pick a writable base dir on Kaggle
KAGGLE = Path("/kaggle/working").exists()
BASE = Path("/kaggle/working") if KAGGLE else Path("/mnt/data")
BASE.mkdir(parents=True, exist_ok=True)

# 2) Remap CONFIG paths to Kaggle's working dir when needed
try:
    CONFIG
except NameError:
    CONFIG = {}

def _p(name): return str(BASE / name)

CONFIG.update({
    "RUN_UI": False,            # Widgets may not render on Kaggle; set True if you know they do
    "RUN_PIPELINE": False,
    "RUN_SYNTH": False,         # <- set True for the synthetic demo
    "SAVE_SYNTH": False,
    "RUN_MEDS": False,          # <- set True for the meds demo
    "EQUIPMENT_STATUS_PATH": _p("equipment_status.csv"),
    "EQUIPMENT_MOVES_LOG_PATH": _p("moves_log.csv"),
    "SOP_REGISTRY_PATH": _p("sop_registry.csv"),
    "QR_OUTPUT_DIR": _p("qrs"),
    "EVENT_LOG_PATH": _p("event_log.jsonl"),
})

# 3) If your real CSVs are attached as a Kaggle Dataset, copy them from /kaggle/input/* to BASE
#    (Adjust the dataset folder name below to whatever you attached.)
INPUT_ROOT = Path("/kaggle/input")
if KAGGLE and INPUT_ROOT.exists():
    # Try common names; skip if already present
    for fname in ["train_DE_full.csv","val_DE_full.csv","test_DE_full.csv","meta.json","kpi_report.json","demo_med_plan.txt"]:
        src = next(INPUT_ROOT.rglob(fname), None)
        dst = BASE / fname
        if src and not dst.exists():
            dst.write_bytes(src.read_bytes())

print("BASE =", BASE)
print("CONFIG['EVENT_LOG_PATH'] =", CONFIG["EVENT_LOG_PATH"])

# 4) Optional: ensure minimal deps (Kaggle usually has sklearn/pandas/numpy preinstalled)
# NOTE: OCR will be paste-only on Kaggle unless tesseract is available.
# If widgets don't render, keep RUN_UI=False and use the print-mode demo cells.


BASE = /kaggle/working
CONFIG['EVENT_LOG_PATH'] = /kaggle/working/event_log.jsonl


In [53]:
# === Phase-2 calculators (extended) + bundle + HL7 paste adapter (single cell, no sidecars) ===
from __future__ import annotations
from typing import Dict, Any, Optional, List, Tuple
from datetime import datetime
import json, re, math

EVENT_LOG_PATH = CONFIG.get("EVENT_LOG_PATH", "/mnt/data/event_log.jsonl")

def _append_event(ev: Dict[str, Any]):
    ev = {"ts": datetime.utcnow().isoformat()+"Z", **ev}
    with open(EVENT_LOG_PATH, "a") as fp:
        fp.write(json.dumps(ev, ensure_ascii=False) + "\n")

# ---------- helpers ----------
def _pick(d: Dict[str, Any], names: List[str], cast=float, default=None):
    for n in names:
        if n in d and d[n] not in (None, ""):
            try:
                return cast(d[n])
            except Exception:
                try:
                    return cast(str(d[n]).replace(",", "."))
                except Exception:
                    pass
    return default
def _bool(v): 
    if isinstance(v, bool): return v
    s=str(v).strip().lower()
    return s in {"1","true","yes","y","ja","oui","on"}
def _safe_round(x, nd=1):
    try: return round(float(x), nd)
    except: return x
def _mmolL_to_mgdl_urea(urea_mmolL):  # GBS expects urea mmol/L
    return float(urea_mmolL)*2.8  # for message only
def _bun_mgdl_to_urea_mmolL(bun_mgdl):
    return float(bun_mgdl)/2.8

# ---------- existing calculators (qSOFA/MEWS/HEART/GRACE/SOFA/D-dimer/Troponin-Δ) ----------
def calc_qsofa(vitals: Dict[str, Any]) -> Dict[str, Any]:
    rr  = _pick(vitals, ["rr","resp_rate","respiratoryrate","af"])
    sbp = _pick(vitals, ["sbp","systolic","rrsys"])
    gcs = _pick(vitals, ["gcs","glasgowcoma","glasgow","gcs_total"], cast=float)
    avpu = str(vitals.get("avpu","")).strip().upper() if vitals.get("avpu") is not None else ""
    altered = (gcs is not None and gcs < 15) or (avpu in {"V","P","U"})
    score = int((rr is not None and rr >= 22)) + int((sbp is not None and sbp <= 100)) + int(altered)
    return {"name":"qSOFA","score":score,"components":{
        "RR>=22": int(rr is not None and rr>=22),
        "SBP<=100": int(sbp is not None and sbp<=100),
        "GCS<15 or AVPU!=A": int(altered)
    }}

def calc_mews(vitals: Dict[str, Any]) -> Dict[str, Any]:
    def rr_s(x):  return 3 if x is not None and x<=8 else (0 if x is not None and 9<=x<=14 else (1 if x and 15<=x<=20 else (2 if x and 21<=x<=29 else (3 if x and x>=30 else 0))))
    def hr_s(x):  return 2 if x is not None and x<=40 else (1 if x and 41<=x<=50 else (0 if x and 51<=x<=100 else (1 if x and 101<=x<=110 else (2 if x and 111<=x<=129 else (3 if x and x>=130 else 0)))))
    def sbp_s(x): return 3 if x is not None and x<=70 else (2 if x and 71<=x<=80 else (1 if x and 81<=x<=100 else (0 if x and 101<=x<=199 else (2 if x and x>=200 else 0))))
    def t_s(x):   return 2 if x is not None and x<=35.0 else (1 if x and 35.1<=x<=36.0 else (0 if x and 36.1<=x<=38.0 else (1 if x and 38.1<=x<=38.5 else (2 if x and x>=38.6 else 0))))
    def avpu_s(x): return {"A":0,"V":1,"P":2,"U":3}.get(str(x).upper()[:1],0)
    rr  = _pick(vitals, ["rr","resp_rate","respiratoryrate","af"])
    hr  = _pick(vitals, ["hr","pulse","herzfrequenz","hf"])
    sbp = _pick(vitals, ["sbp","systolic","rrsys"])
    temp= _pick(vitals, ["temp","temperature","kt"], cast=float)
    avpu= vitals.get("avpu","A")
    comps = {"RR": rr_s(rr), "HR": hr_s(hr), "SBP": sbp_s(sbp), "Temp": t_s(temp), "AVPU": avpu_s(avpu)}
    return {"name":"MEWS","score": int(sum(comps.values())), "components": comps}

def calc_heart(params: Dict[str, Any]) -> Dict[str, Any]:
    age   = _pick(params, ["age","alter"], cast=int)
    hist  = _pick(params, ["heart_history","history_score"], cast=int)
    ecg   = _pick(params, ["heart_ecg","ecg_score"], cast=int)
    risk  = _pick(params, ["heart_risk","riskf_score"], cast=int)
    trop  = _pick(params, ["troponin","hs_trop","hs_troponin","trop"], cast=float)
    uln   = _pick(params, ["troponin_uln","trop_uln","troponin_ref_high"], cast=float)
    age_s = 2 if (age is not None and age>=65) else (1 if (age is not None and 45<=age<=64) else 0)
    ratio = (trop/uln) if (trop is not None and uln) else None
    trop_s = 2 if (ratio is not None and ratio>3) else (1 if (ratio is not None and 1<ratio<=3) else (0 if ratio is not None else None))
    comps = {"History": hist if hist is not None else None,
             "ECG": ecg if ecg is not None else None,
             "Age": age_s if age is not None else None,
             "Risk": risk if risk is not None else None,
             "Troponin": trop_s if trop_s is not None else None}
    total = sum(v for v in comps.values() if v is not None)
    return {"name":"HEART","score": total, "components": comps, "note":"coarse; clinician inputs required for History/ECG/Risk."}

def calc_grace_coarse(params: Dict[str, Any]) -> Dict[str, Any]:
    age   = _pick(params, ["age","alter"], cast=int)
    hr    = _pick(params, ["hr","pulse","herzfrequenz","hf"])
    sbp   = _pick(params, ["sbp","systolic","rrsys"])
    crea  = _pick(params, ["creatinine","crea","kr","kreatinin"])
    score = 0
    if age is not None: score += (0 if age<40 else 20 if age<60 else 40 if age<80 else 60)
    if hr  is not None: score += (0 if hr<70  else 10 if hr<90  else 20 if hr<110 else 30 if hr<150 else 40)
    if sbp is not None: score += (40 if sbp<80 else 30 if sbp<100 else 10 if sbp<120 else 0)
    if crea is not None: score += (0 if crea<1.2 else 10 if crea<2.0 else 20 if crea<3.0 else 30)
    return {"name":"GRACE_coarse","score": int(score), "components":{"age":age,"hr":hr,"sbp":sbp,"creatinine":crea}, "note":"coarse proxy"}

def calc_sofa_minimal(vitals: Dict[str, Any], labs: Dict[str, Any]) -> Dict[str, Any]:
    pf=None
    pao2=_pick(labs,["pao2","paO2","PaO2"]); fio2=_pick(labs,["fio2","FiO2","FIO2"])
    if pao2 is not None and fio2: 
        try: pf=float(pao2)/float(fio2)
        except: pf=None
    plate=_pick(labs,["platelets","plt","thrombozyten"]); bili=_pick(labs,["bilirubin","bili","total_bilirubin"])
    mapv=_pick(vitals,["map","meanbp","mad"]); gcs=_pick(vitals,["gcs","glasgow"]); crea=_pick(labs,["creatinine","crea","kr","kreatinin"])
    sofa=0; comps={}
    if pf is not None:        s=4 if pf<100 else 3 if pf<200 else 2 if pf<300 else 1 if pf<400 else 0; sofa+=s; comps["resp"]=s
    if plate is not None:     s=4 if plate<20 else 3 if plate<50 else 2 if plate<100 else 1 if plate<150 else 0; sofa+=s; comps["coag"]=s
    if bili  is not None:     s=4 if bili>=12 else 3 if bili>=6 else 2 if bili>=2 else 1 if bili>=1.2 else 0; sofa+=s; comps["liver"]=s
    if mapv  is not None:     s=1 if mapv<70 else 0; sofa+=s; comps["cardio"]=s
    if gcs   is not None:     s=4 if gcs<6 else 3 if gcs<10 else 2 if gcs<13 else 1 if gcs<15 else 0; sofa+=s; comps["cns"]=s
    if crea  is not None:     s=4 if crea>=5 else 3 if crea>=3.5 else 2 if crea>=2 else 1 if crea>=1.2 else 0; sofa+=s; comps["renal"]=s
    return {"name":"SOFA_minimal","score": int(sofa), "components": comps, "note":"partial; uses available values only"}

def assess_d_dimer(value: Optional[float], unit: Optional[str], age: Optional[int], pregnant: Optional[bool]=False) -> Dict[str, Any]:
    if value is None: return {"available": False}
    unit = (unit or "").lower().strip()
    val_ug = float(value)*1000.0 if "mg/l" in unit else float(value)  # normalize to μg/L FEU
    thr = 500.0
    if age is not None and age>50 and not pregnant: thr = float(age)*10.0
    return {"available": True, "value_ug_per_l": val_ug, "thr_ug_per_l": thr, "ok_below_thr": bool(val_ug < thr), "unit_norm":"μg/L FEU", "notes": ("pregnancy disables age-adjustment" if pregnant else "")}

def assess_troponin_delta(series: List[Tuple[Optional[datetime], float, str]]) -> Dict[str, Any]:
    if not series or len(series)<2: return {"available": False}
    try: series = sorted(series, key=lambda x: (x[0] or datetime.min))
    except: pass
    prev_t, prev_v, prev_u = series[-2]; curr_t, curr_v, curr_u = series[-1]
    def to_ng_per_l(v,u):
        u=(u or "").lower()
        if "ng/l" in u: return float(v)
        if "pg/ml" in u: return float(v)    # pg/mL == ng/L
        if "µg/l" in u or "ug/l" in u: return float(v)*1000.0
        return float(v)
    pv = to_ng_per_l(prev_v, prev_u); cv = to_ng_per_l(curr_v, curr_u)
    delta_abs = cv - pv
    delta_pct = (abs(delta_abs)/pv*100.0) if pv!=0 else None
    flag = (abs(delta_abs) >= 51.0) or (delta_pct is not None and delta_pct >= 20.0)
    return {"available": True, "prev": pv, "curr": cv, "delta_abs": delta_abs, "delta_pct": delta_pct, "flag": bool(flag)}

# ---------- NEW: Wells (PE/DVT), PERC, PESI/sPESI, SIRS, Sepsis-3 screen, Marburg, GBS, Child-Pugh, Ca-corr, Anion Gap ----------
def calc_wells_pe(p: Dict[str, Any]) -> Dict[str, Any]:
    # Points per Wells PE (two-tier or three-tier). Sources: Wells PE criteria. :contentReference[oaicite:0]{index=0}
    pts = 0.0
    comps = {}
    def add(name, cond, val): 
        nonlocal pts; pts += val; comps[name]=int(bool(cond))
    add("DVT_signs", _bool(p.get("dvt_signs")), 3.0)
    add("PE_most_likely", _bool(p.get("pe_most_likely")), 3.0)
    add("HR>100", (_pick(p, ["hr","pulse","hf"], float) or 0) > 100, 1.5)
    add("immobilization_or_surgery_4w", _bool(p.get("immobilized")) or _bool(p.get("recent_surgery_4w")), 1.5)
    add("prev_DVT_PE", _bool(p.get("prev_vte")), 1.5)
    add("hemoptysis", _bool(p.get("hemoptysis")), 1.0)
    add("malignancy", _bool(p.get("cancer_active")), 1.0)
    tier3 = "high" if pts>6 else ("moderate" if 2<=pts<=6 else "low")
    tier2 = "likely" if pts>4 else "unlikely"
    return {"name":"WELLS_PE","score": pts, "tiers":{"three":tier3,"two":tier2}, "components": comps}

def calc_wells_dvt(p: Dict[str, Any]) -> Dict[str, Any]:
    # Wells DVT (two-tier or three-tier). Sources. :contentReference[oaicite:1]{index=1}
    pts=0; comps={}
    def add(name, cond, val): 
        nonlocal pts; pts += val; comps[name]=int(bool(cond))
    add("active_cancer", _bool(p.get("cancer_active")), 1)
    add("paralysis_or_plaster", _bool(p.get("paresis")) or _bool(p.get("plaster_cast")), 1)
    add("bedridden_3d_or_surgery_12w", _bool(p.get("bedridden_3d")) or _bool(p.get("surgery_12w")), 1)
    add("tenderness_deep_veins", _bool(p.get("deep_vein_tenderness")), 1)
    add("entire_leg_swollen", _bool(p.get("entire_leg_swollen")), 1)
    add("calf_swelling_>3cm", _bool(p.get("calf_swelling_gt3cm")), 1)
    add("pitting_edema_symptomatic_leg", _bool(p.get("pitting_edema")), 1)
    add("collateral_superficial_veins", _bool(p.get("collateral_nonvaricose")), 1)
    add("previous_dvt", _bool(p.get("prev_dvt")), 1)
    add("alt_diagnosis_as_likely", _bool(p.get("alt_dx_as_likely")), -2)
    tier3 = "high" if pts>=3 else ("moderate" if 1<=pts<=2 else "low")
    tier2 = "likely" if pts>=2 else "unlikely"
    return {"name":"WELLS_DVT","score": int(pts), "tiers":{"three":tier3,"two":tier2}, "components": comps}

def calc_perc(p: Dict[str, Any]) -> Dict[str, Any]:
    # PERC (all must be negative to rule out in low pretest prob). :contentReference[oaicite:2]{index=2}
    crit = {
        "age<50": (_pick(p,["age"],int) or 10) < 50,
        "hr<100": (_pick(p,["hr","hf","pulse"],float) or 0) < 100,
        "sao2>=95": (_pick(p,["sao2","spo2","o2sat"],float) or 0) >= 95,
        "no_hemoptysis": not _bool(p.get("hemoptysis")),
        "no_estrogen": not _bool(p.get("estrogen_use")),
        "no_surgery_or_trauma_4w": not (_bool(p.get("recent_surgery_4w")) or _bool(p.get("recent_trauma_4w"))),
        "no_prior_vte": not _bool(p.get("prev_vte")),
        "no_unilateral_leg_swelling": not _bool(p.get("unilateral_leg_swelling")),
    }
    all_neg = all(crit.values())
    return {"name":"PERC","passed": bool(all_neg), "components": {k:int(v) for k,v in crit.items()}}

def calc_pesi(p: Dict[str, Any]) -> Dict[str, Any]:
    # Original PESI. Summation of age (years) + points for variables. :contentReference[oaicite:3]{index=3}
    age = _pick(p,["age"],int) or 0
    male = 10 if (str(p.get("sex") or p.get("geschlecht") or "").upper().startswith("M")) else 0
    cancer = 30 if _bool(p.get("cancer_active")) else 0
    hf = 10 if _bool(p.get("heart_failure")) else 0
    lung = 10 if (_bool(p.get("copd")) or _bool(p.get("chronic_lung_disease"))) else 0
    hr = 20 if (_pick(p,["hr","hf","pulse"],float) or 0) >=110 else 0
    sbp = 30 if (_pick(p,["sbp","systolic"],float) or 200) < 100 else 0
    rr = 20 if (_pick(p,["rr","resp_rate"],float) or 0) >=30 else 0
    temp = 20 if (_pick(p,["temp","temperature"],float) or 37) < 36 else 0
    altered = 60 if (_bool(p.get("altered_mental_status")) or (_pick(p,["gcs"],float) or 15) < 15) else 0
    sat = 20 if (_pick(p,["sao2","spo2","o2sat"],float) or 100) < 90 else 0
    score = age + male + cancer + hf + lung + hr + sbp + rr + temp + altered + sat
    # risk class I–V via cutpoints
    if   score<=65: klass="I"
    elif score<=85: klass="II"
    elif score<=105: klass="III"
    elif score<=125: klass="IV"
    else: klass="V"
    return {"name":"PESI","score": int(score), "class": klass}

def calc_spesi(p: Dict[str, Any]) -> Dict[str, Any]:
    # sPESI: 1 pt each. Age>80, cancer, chronic cardiopulmonary dz, HR≥110, SBP<100, O2<90%. :contentReference[oaicite:4]{index=4}
    comps = {
        "age>80": int((_pick(p,["age"],int) or 0) > 80),
        "cancer": int(_bool(p.get("cancer_active"))),
        "cardiopulmonary_disease": int(_bool(p.get("heart_failure")) or _bool(p.get("copd")) or _bool(p.get("chronic_lung_disease"))),
        "hr>=110": int((_pick(p,["hr","hf","pulse"],float) or 0) >= 110),
        "sbp<100": int((_pick(p,["sbp","systolic"],float) or 200) < 100),
        "o2<90": int((_pick(p,["sao2","spo2","o2sat"],float) or 100) < 90),
    }
    score = sum(comps.values())
    return {"name":"sPESI","score": int(score)}

def calc_sirs(p: Dict[str, Any]) -> Dict[str, Any]:
    # SIRS criteria (≥2). :contentReference[oaicite:5]{index=5}
    crit = {
        "temp>38_or_<36": int(((_pick(p,["temp","temperature"],float) or 37)>38) or ((_pick(p,["temp","temperature"],float) or 37)<36)),
        "hr>90": int((_pick(p,["hr","hf","pulse"],float) or 0) > 90),
        "rr>20_or_paco2<32": int(((_pick(p,["rr","resp_rate"],float) or 0) > 20) or ((_pick(p,["paco2"],float) or 100) < 32)),
        "wbc>12_or_<4_or_bands>10pct": int(((_pick(p,["wbc"],float) or 7) > 12) or ((_pick(p,["wbc"],float) or 7) < 4) or ((_pick(p,["bands_pct"],float) or 0) > 10)),
    }
    return {"name":"SIRS","score": int(sum(crit.values())), "components": crit}

def sepsis3_screen(vitals: Dict[str, Any], labs: Dict[str, Any], context: Dict[str, Any], sofa_min: Dict[str, Any], qsofa: Dict[str, Any]) -> Dict[str, Any]:
    # Sepsis-3: suspected infection + SOFA increase ≥2 (we expose a screening surrogate). Septic shock: MAP<65 & lactate>2 & vasopressors. :contentReference[oaicite:6]{index=6}
    suspected_infection = _bool(context.get("suspected_infection"))
    sofa_score = sofa_min["score"]
    qsofa_score = qsofa["score"]
    mapv = _pick(vitals,["map","meanbp","mad"])
    lact = _pick(labs,["lactate","laktat"])
    on_pressors = _bool(context.get("vasopressors"))
    septic_shock = bool((mapv is not None and mapv<65) and (lact is not None and lact>2) and on_pressors)
    sepsis_flag = bool(suspected_infection and sofa_score>=2)
    return {"name":"SEPSIS3_SCREEN","suspected_infection": suspected_infection, "sofa_min": sofa_score, "qsofa": qsofa_score, "septic_shock": septic_shock, "sepsis_flag": sepsis_flag}

def calc_marburg(params: Dict[str, Any]) -> Dict[str, Any]:
    sex = (str(params.get("sex") or params.get("geschlecht") or "")[:1]).upper()
    age = _pick(params, ["age","alter"], cast=int)
    vasc = _bool(params.get("vasc_disease") or params.get("known_vascular_disease") or params.get("kvd"))
    exertional = _bool(params.get("exertional") or params.get("pain_with_exertion") or params.get("belastungsabhaengig"))
    patient_assumes = _bool(params.get("patient_assumes_cardiac") or params.get("pt_thinks_cardiac"))
    palp_repro = params.get("palpation_reproducible")
    not_reproducible = (palp_repro is False)
    age_sex = ((sex == "M" and age is not None and age >= 55) or (sex == "F" and age is not None and age >= 65))
    comps = {"age_sex": int(bool(age_sex)),
             "known_vascular_disease": int(vasc),
             "pain_worse_with_exertion": int(exertional),
             "patient_assumes_cardiac": int(patient_assumes),
             "pain_not_reproducible_by_palpation": int(bool(not_reproducible))}
    score = int(sum(comps.values()))
    return {"name": "MARBURG", "score": score, "components": comps}

def calc_gbs(p: Dict[str, Any]) -> Dict[str, Any]:
    # Glasgow-Blatchford (upper GI bleed). Uses urea mmol/L (will convert from BUN mg/dL if given). :contentReference[oaicite:7]{index=7}
    # Inputs: urea_mmolL or bun_mgdl; hb_gL or hb_gdl; sbp_mmHg; hr; melena; syncope; hepatic_dz; heart_failure
    urea = _pick(p,["urea_mmol_l","urea"],float)
    bun = _pick(p,["bun_mg_dl","bun"],float)
    if urea is None and bun is not None: urea = _bun_mgdl_to_urea_mmolL(bun)
    hb_gL = _pick(p,["hb_g_per_l","hb_g_l"],float)
    if hb_gL is None:
        hb_gdl = _pick(p,["hb_g_dl","hb"],float)
        if hb_gdl is not None: hb_gL = hb_gdl*10.0
    sbp = _pick(p,["sbp","systolic"],float)
    hr  = _pick(p,["hr","pulse","hf"],float)
    male = str(p.get("sex") or p.get("geschlecht") or "").upper().startswith("M")
    score=0
    # Urea
    if urea is not None:
        score += 2 if 6.5<=urea<=7.9 else 0
        score += 3 if 8.0<=urea<=9.9 else 0
        score += 4 if 10.0<=urea<=25.0 else 0
        score += 6 if urea>25.0 else 0
    # Hemoglobin
    if hb_gL is not None:
        if male:
            score += 1 if 120<=hb_gL<=129 else 0
            score += 3 if 100<=hb_gL<=119 else 0
            score += 6 if hb_gL<100 else 0
        else:
            score += 1 if 100<=hb_gL<=119 else 0
            score += 6 if hb_gL<100 else 0
    # SBP
    if sbp is not None:
        score += 1 if 100<=sbp<=109 else 0
        score += 2 if 90<=sbp<=99 else 0
        score += 3 if sbp<90 else 0
    # Other markers
    score += 1 if (hr is not None and hr>=100) else 0
    score += 1 if _bool(p.get("melena")) else 0
    score += 2 if _bool(p.get("syncope")) else 0
    score += 2 if _bool(p.get("hepatic_disease")) else 0
    score += 2 if _bool(p.get("cardiac_failure")) else 0
    return {"name":"GBS","score": int(score)}

def calc_child_pugh(p: Dict[str, Any]) -> Dict[str, Any]:
    # Child-Pugh A/B/C. Uses bilirubin mg/dL, albumin g/dL, INR, ascites none/mild/mod, encephalopathy 0/1-2/3-4. :contentReference[oaicite:8]{index=8}
    bili = _pick(p,["bilirubin","bili"],float)
    alb  = _pick(p,["albumin","alb"],float)
    inr  = _pick(p,["inr"],float)
    asc  = str(p.get("ascites") or "").lower()   # none/mild/moderate/severe
    ence = str(p.get("encephalopathy") or "").lower()  # none/1-2/3-4
    sc=0; comps={}
    if bili is not None:
        s = 1 if bili<2 else (2 if bili<=3 else 3); sc+=s; comps["bilirubin"]=s
    if alb is not None:
        s = 1 if alb>3.5 else (2 if alb>=2.8 else 3); sc+=s; comps["albumin"]=s
    if inr is not None:
        s = 1 if inr<1.7 else (2 if inr<=2.3 else 3); sc+=s; comps["inr"]=s
    if asc:
        s = 1 if asc.startswith("n") else (2 if asc.startswith(("mild","slight")) else 3); sc+=s; comps["ascites"]=s
    if ence:
        if ence in {"none","0"}: s=1
        elif any(x in ence for x in ["1","2","grade 1","grade 2","i","ii"]): s=2
        else: s=3
        sc+=s; comps["encephalopathy"]=s
    klass = "A" if sc<=6 else ("B" if sc<=9 else "C")
    return {"name":"CHILD_PUGH","score": int(sc), "class": klass, "components": comps}

def corrected_calcium(total_ca, albumin, units="mg/dL", normal_alb=None):
    # Corrected calcium by albumin (Payne). mg/dL: Ca_corr = Ca + 0.8*(4 - Alb[g/dL]); mmol/L: Ca + 0.02*(40 - Alb[g/L]). :contentReference[oaicite:9]{index=9}
    if total_ca is None or albumin is None: return None
    units = units.lower()
    if "mmol" in units:
        alb_gl = albumin if albumin>10 else albumin*10.0  # accept g/L directly; if g/dL given (~2-5), convert to g/L
        normal = 40.0 if normal_alb is None else float(normal_alb)
        return float(total_ca) + 0.02*(normal - alb_gl)
    # default mg/dL
    normal = 4.0 if normal_alb is None else float(normal_alb)
    return float(total_ca) + 0.8*(normal - float(albumin))

def anion_gap(na, cl, hco3, k=None, albumin_gdl=None):
    # AG = Na (+K optional) - (Cl + HCO3); Albumin-corrected AG ≈ AG + 2.5*(4 - albumin[g/dL]). :contentReference[oaicite:10]{index=10}
    if na is None or cl is None or hco3 is None: return None
    ag = (float(na) + (float(k) if k is not None else 0.0)) - (float(cl) + float(hco3))
    ag_corr = ag + (2.5*(4.0 - float(albumin_gdl))) if albumin_gdl is not None else None
    return {"ag": _safe_round(ag,1), "ag_albumin_corrected": _safe_round(ag_corr,1) if ag_corr is not None else None}

# ---------- NEW: Pregnancy pathways ----------
def years_pregnancy_pathway(p: Dict[str, Any]) -> Dict[str, Any]:
    # Pregnancy-adapted YEARS: Items: DVT signs, hemoptysis, PE most likely.
    # D-dimer FEU threshold: 1000 μg/L if 0 items; 500 μg/L if ≥1 item.
    # If DVT symptoms present, CUS first; positive => treat; negative => continue algorithm. :contentReference[oaicite:11]{index=11}
    items = {
        "dvt_signs": _bool(p.get("dvt_signs")),
        "hemoptysis": _bool(p.get("hemoptysis")),
        "pe_most_likely": _bool(p.get("pe_most_likely")),
    }
    n_items = sum(int(v) for v in items.values())
    d_feu = _pick(p,["d_dimer_feu_ug_l","d_dimer","ddimer"],float)
    cus_done = _bool(p.get("cus_done"))
    cus_positive = _bool(p.get("cus_positive")) if cus_done else None
    thr = 1000.0 if n_items==0 else 500.0
    ruled_out = (d_feu is not None and d_feu < thr) and (not (_bool(p.get("dvt_signs")) and cus_done and cus_positive))
    next_step = "No imaging (PE ruled out by YEARS pregnancy)" if ruled_out else (
        "Perform CUS first (DVT symptoms present)" if (_bool(p.get("dvt_signs")) and not cus_done) else
        "CTPA/VQ per local protocol")
    return {"name":"YEARS_PREGNANCY","items":items,"n_items":n_items,"d_dimer_ug_l":d_feu,"threshold_ug_l":thr,
            "cus_done":cus_done,"cus_positive":cus_positive,"ruled_out":bool(ruled_out),"recommended_next_step":next_step}

def left_score(p: Dict[str, Any]) -> Dict[str, Any]:
    # LEFt rule for DVT in pregnancy: Left leg symptoms, Edema (calf diff ≥2 cm), First-trimester. 0–3. :contentReference[oaicite:12]{index=12}
    left = _bool(p.get("left_leg_symptoms"))
    edema2 = _bool(p.get("calf_diff_ge_2cm"))
    first_tri = str(p.get("pregnancy_trimester") or "").strip() in {"1","first","i"}
    score = int(left) + int(edema2) + int(first_tri)
    return {"name":"LEFT_PREG_DVT","score": score, "components":{"left":int(left),"edema>=2cm":int(edema2),"first_trimester":int(first_tri)}}

# ---------- HL7 paste adapter (Troponin & D-dimer) ----------
def parse_hl7_labs(hl7_text: str) -> Dict[str, Any]:
    troponin, ddimer = [], []
    lines = re.split(r'[\r\n]+', hl7_text.strip())
    for ln in lines:
        if not ln.strip(): continue
        parts = ln.split("|")
        obx3 = parts[3] if len(parts)>3 else ""
        obx5 = parts[5] if len(parts)>5 else ""
        obx6 = parts[6] if len(parts)>6 else ""
        obx14= parts[14] if len(parts)>14 else ""
        name = (obx3 or ln); val = obx5; unit = obx6
        ts = None
        mdt = re.search(r'(\d{8})(\d{6})?', obx14)
        if mdt:
            try: ts = datetime.strptime(mdt.group(1)+(mdt.group(2) or "000000"), "%Y%m%d%H%M%S")
            except: ts = None
        v = None
        try: v = float(str(val).replace(",", ".").strip())
        except:
            mnum = re.search(r'[-+]?\d*\.?\d+', str(val).replace(",", "."))
            if mnum:
                try: v = float(mnum.group(0))
                except: v = None
        if not unit:
            munit = re.search(r'\[(.*?)\]', name)
            if munit: unit = munit.group(1)
        n = name.upper()
        if ("TROP" in n or "TROPONIN" in n) and v is not None: troponin.append((ts, v, unit or "ng/L"))
        if ("D-DIMER" in n or "DDIMER" in n or "D DIMER" in n) and v is not None: ddimer.append((ts, v, unit or "μg/L FEU"))
        if v is None:
            if re.search(r'trop', ln, re.I):
                m = re.search(r'([-+]?\d*\.?\d+)', ln.replace(",", ".")); 
                if m: troponin.append((ts, float(m.group(1)), "ng/L"))
            if re.search(r'd[\-\s]?dimer', ln, re.I):
                m = re.search(r'([-+]?\d*\.?\d+)', ln.replace(",", ".")); 
                if m: ddimer.append((ts, float(m.group(1)), "μg/L FEU"))
    return {"troponin": troponin, "d_dimer": ddimer}

# ---------- bundle ----------
def phase2_bundle(vitals: Dict[str, Any], labs: Dict[str, Any], context: Optional[Dict[str, Any]]=None) -> Dict[str, Any]:
    context = context or {}
    age = _pick({**vitals, **labs, **context}, ["age","alter"], cast=int)
    pregnant = bool(context.get("pregnant", False))
    # base scores
    s_qsofa = calc_qsofa(vitals)
    s_mews  = calc_mews(vitals)
    s_grace = calc_grace_coarse({**vitals, **labs})
    s_heart = calc_heart({**vitals, **labs})
    s_sofa  = calc_sofa_minimal(vitals, labs)
    # labs: d-dimer + troponin Δ
    d_val = _pick(labs, ["d_dimer","ddimer","d-dimer","d_dimer_value","d_dimer_feu_ug_l"], cast=float)
    d_unit = labs.get("d_dimer_unit") or labs.get("ddimer_unit") or labs.get("unit_d_dimer") or ("μg/L FEU" if d_val is not None else None)
    d_assess = assess_d_dimer(d_val, d_unit, age, pregnant) if d_val is not None else {"available": False}
    series = []
    if isinstance(labs.get("troponin_series"), (list, tuple)):
        for item in labs["troponin_series"]:
            if isinstance(item, dict):
                ts=None
                if item.get("time"):
                    try: ts = datetime.fromisoformat(str(item["time"]).replace("Z",""))
                    except: ts=None
                series.append((ts, _pick(item, ["value","val"], cast=float), item.get("unit","ng/L")))
    elif "troponin_prev" in labs or "troponin_curr" in labs:
        prev = _pick(labs, ["troponin_prev","hs_trop_prev"]); curr = _pick(labs, ["troponin_curr","hs_trop_curr"])
        if prev is not None and curr is not None:
            series = [(None, prev, labs.get("troponin_unit","ng/L")), (None, curr, labs.get("troponin_unit","ng/L"))]
    t_assess = assess_troponin_delta(series) if series else {"available": False}
    # NEW: additional calculators
    wells_pe   = calc_wells_pe({**vitals, **labs, **context})
    wells_dvt  = calc_wells_dvt({**vitals, **labs, **context})
    perc       = calc_perc({**vitals, **labs, **context})
    pesi       = calc_pesi({**vitals, **labs, **context})
    spesi      = calc_spesi({**vitals, **labs, **context})
    sirs       = calc_sirs({**vitals, **labs})
    sepsis3    = sepsis3_screen(vitals, labs, context, s_sofa, s_qsofa)
    marburg    = calc_marburg({**vitals, **labs, **context})
    gbs        = calc_gbs({**vitals, **labs, **context})
    childpugh  = calc_child_pugh({**vitals, **labs, **context})
    ca_corr    = corrected_calcium(_pick(labs,["calcium_mg_dl","calcium","ca"],float),
                                   _pick(labs,["albumin_g_dl","albumin","alb"],float),
                                   units="mg/dL")
    ag         = anion_gap(_pick(labs,["na","sodium"],float), _pick(labs,["cl","chloride"],float), _pick(labs,["hco3","bicarbonate"],float),
                           k=_pick(labs,["k","potassium"],float), albumin_gdl=_pick(labs,["albumin_g_dl","albumin","alb"],float))
    # Pregnancy-specific PE/DVT pathways (optional)
    years_preg = years_pregnancy_pathway({**vitals, **labs, **context}) if pregnant else None
    left_preg  = left_score({**vitals, **labs, **context}) if pregnant else None

    ones = []
    ones.append(f"qSOFA={s_qsofa['score']}  MEWS={s_mews['score']}  GRACE(coarse)={s_grace['score']}")
    ones.append(f"HEART={s_heart['score']}  SOFA(min)={s_sofa['score']}")
    if d_assess.get("available"):
        ones.append(f"D-dimer {int(d_assess['value_ug_per_l'])} vs thr {int(d_assess['thr_ug_per_l'])} μg/L → {'OK' if d_assess['ok_below_thr'] else 'High'}")
    if t_assess.get("available"):
        da=_safe_round(t_assess['delta_abs'],1); dp=_safe_round(t_assess['delta_pct'],1) if t_assess['delta_pct'] is not None else None
        ones.append(f"Troponin Δ {da} ng/L ({dp}%) → {'FLAG' if t_assess['flag'] else 'ok'}")
    ones.append(f"Wells-PE={_safe_round(wells_pe['score'],1)} ({wells_pe['tiers']['two']})  Wells-DVT={wells_dvt['score']} ({wells_dvt['tiers']['two']})")
    ones.append(f"PERC={'pass' if perc['passed'] else 'fail'}  sPESI={spesi['score']}  PESI={pesi['class']}/{pesi['score']}")
    ones.append(f"SIRS={sirs['score']}  Sepsis3: {'YES' if sepsis3['sepsis_flag'] else 'no'}  Shock: {'YES' if sepsis3['septic_shock'] else 'no'}")
    ones.append(f"MARBURG={marburg['score']}/5  GBS={gbs['score']}")
    if childpugh["components"]:
        ones.append(f"Child-Pugh {childpugh['class']} ({childpugh['score']})")
    if ca_corr is not None:
        ones.append(f"Corrected Ca={_safe_round(ca_corr,2)} mg/dL")
    if ag is not None:
        ab = f", AGcorr={ag['ag_albumin_corrected']}" if ag.get("ag_albumin_corrected") is not None else ""
        ones.append(f"Anion gap={ag['ag']}{ab}")
    if years_preg:
        verdict = "ruled out" if years_preg["ruled_out"] else "needs imaging"
        ones.append(f"YEARS(preg): items={years_preg['n_items']} D-dimer<{int(years_preg['threshold_ug_l'])}? → {verdict}")
    if left_preg:
        ones.append(f"LEFt(preg DVT)={left_preg['score']}/3")

    out = {
        "scores":{"qSOFA":s_qsofa,"MEWS":s_mews,"GRACE_coarse":s_grace,"HEART":s_heart,"SOFA_min":s_sofa,
                  "WELLS_PE":wells_pe,"WELLS_DVT":wells_dvt,"PERC":perc,"PESI":pesi,"sPESI":spesi,"SIRS":sirs,
                  "SEPSIS3_SCREEN":sepsis3,"MARBURG":marburg,"GBS":gbs,"CHILD_PUGH":childpugh},
        "chem":{"corrected_calcium_mg_dl": ca_corr, "anion_gap": ag},
        "pregnancy":{"YEARS": years_preg, "LEFT": left_preg} if pregnant else {},
        "labs":{"d_dimer":{"value":d_val,"unit":d_unit,"assessment":d_assess},"troponin_delta":t_assess},
        "one_liners": ones
    }
    return out

def process_hl7_and_log(hl7_text: str, patient_id: Optional[str]=None, age: Optional[int]=None, pregnant: Optional[bool]=False):
    parsed = parse_hl7_labs(hl7_text)
    if parsed["d_dimer"]:
        ts,v,u = parsed["d_dimer"][-1]
        assess = assess_d_dimer(v,u,age,pregnant)
        _append_event({"type":"d_dimer_assess","patient_id":patient_id,"value":v,"unit":u,"assessment":assess})
    if len(parsed["troponin"])>=2:
        series = parsed["troponin"][-2:]
        assess = assess_troponin_delta(series)
        _append_event({"type":"troponin_delta","patient_id":patient_id,
                       "series":[{"time":(t.isoformat()+'Z' if t else None),"value":v,"unit":u} for t,v,u in series],
                       "assessment":assess})
    elif len(parsed["troponin"])==1:
        t,v,u = parsed["troponin"][-1]
        _append_event({"type":"troponin_single","patient_id":patient_id,"time":(t.isoformat()+'Z' if t else None),"value":v,"unit":u})

# ---------- optional UI (works when RUN_UI=True) ----------
if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W, pandas as pd
        vitals_in = W.Textarea(value='{"rr":24,"sbp":95,"gcs":14,"hr":120,"temp":38.6,"avpu":"V","map":65,"age":68,"sex":"M"}',
                               description="Vitals JSON", layout=W.Layout(width="100%", height="90px"))
        labs_in = W.Textarea(value='{"d_dimer":780,"d_dimer_unit":"μg/L FEU","troponin_series":[{"time":"2025-08-20T05:00:00Z","value":18,"unit":"ng/L"},{"time":"2025-08-20T09:00:00Z","value":78,"unit":"ng/L"}],"creatinine":1.8,"platelets":95,"albumin":2.8,"calcium":7.9,"na":138,"cl":105,"hco3":20}',
                             description="Labs JSON", layout=W.Layout(width="100%", height="130px"))
        ctx_in = W.Textarea(value='{"pregnant": false, "suspected_infection": true, "vasopressors": false, "pe_most_likely": true, "dvt_signs": false}',
                            description="Context JSON", layout=W.Layout(width="100%", height="80px"))
        pid_in = W.Text(description="Patient ID", value="ED-DEMO-001")
        btn = W.Button(description="Compute + Log")
        hl7_in = W.Textarea(value='''OBX|1|NM|D-DIMER||780|ug/L|0-500||H||F|||20250820090000
OBX|2|NM|TROPONIN T||78|ng/L|||H||F|||20250820090000
OBX|3|NM|TROPONIN T||18|ng/L|||N||F|||20250820050000''',
                            description="HL7 paste", layout=W.Layout(width="100%", height="130px"))
        btn_hl7 = W.Button(description="Parse HL7 + Log")
        out = W.Output()

        def on_click(_):
            with out:
                out.clear_output()
                try:
                    vitals = json.loads(vitals_in.value or "{}")
                    labs = json.loads(labs_in.value or "{}")
                    ctx  = json.loads(ctx_in.value or "{}")
                except Exception as e:
                    print("[parse error]", e); return
                bundle = phase2_bundle(vitals, labs, ctx)
                _append_event({"type":"phase2_bundle","patient_id": pid_in.value, "bundle": bundle})
                print("\n".join(bundle["one_liners"]))
                print("\nLogged to:", EVENT_LOG_PATH)

        def on_hl7(_):
            with out:
                out.clear_output()
                process_hl7_and_log(hl7_in.value, patient_id=pid_in.value,
                                    age=_pick(json.loads(ctx_in.value or "{}"),["age"],int),
                                    pregnant=json.loads(ctx_in.value or "{}").get("pregnant", False))
                print("HL7 processed and logged. See", EVENT_LOG_PATH)

        btn.on_click(on_click); btn_hl7.on_click(on_hl7)
        display(W.VBox([W.HTML("<b>Phase-2 calculators (extended) + HL7 paste</b>"),
                        W.HBox([pid_in]),
                        W.HBox([vitals_in, labs_in]),
                        ctx_in, btn,
                        W.HTML("<b>HL7 Labs (Troponin / D-Dimer)</b>"), hl7_in, btn_hl7, out]))
    except Exception as e:
        print("Phase-2 UI unavailable:", e)
else:
    # text-mode smoke (Kaggle-safe)
    vit = {"rr":24,"sbp":95,"gcs":14,"hr":120,"temp":38.6,"avpu":"V","map":65,"age":68,"sex":"M","sao2":97}
    lab = {"d_dimer":780,"d_dimer_unit":"μg/L FEU","troponin_series":[{"time":"2025-08-20T05:00:00Z","value":18,"unit":"ng/L"},{"time":"2025-08-20T09:00:00Z","value":78,"unit":"ng/L"}],
           "creatinine":1.8,"platelets":95,"albumin":2.8,"calcium":7.9,"na":138,"cl":105,"hco3":20}
    ctx = {"pregnant": False, "suspected_infection": True, "vasopressors": False, "pe_most_likely": True, "dvt_signs": False}
    bundle = phase2_bundle(vit, lab, ctx)
    _append_event({"type":"phase2_bundle_smoke","bundle": bundle})
    print("\n".join(bundle["one_liners"]))
    print("Logged to:", EVENT_LOG_PATH)


qSOFA=3  MEWS=8  GRACE(coarse)=110
HEART=2  SOFA(min)=5
D-dimer 780 vs thr 680 μg/L → High
Troponin Δ 60.0 ng/L (333.3%) → FLAG
Wells-PE=12.5 (likely)  Wells-DVT=7 (likely)
PERC=fail  sPESI=2  PESI=V/188
SIRS=3  Sepsis3: YES  Shock: no
MARBURG=1/5  GBS=3
Child-Pugh A (2)
Corrected Ca=8.86 mg/dL
Anion gap=13.0, AGcorr=16.0
Logged to: /mnt/data/event_log.jsonl


In [30]:
CONFIG["RUN_SYNTH"] = True   # runs the realistic synthetic ML pipeline
CONFIG["RUN_MEDS"]  = True   # enables medication-plan parsing (paste-only if no OCR)
CONFIG["RUN_UI"]    = False  # keep off if widgets don't render in your Kaggle session


In [31]:

# CONFIG bootstrap with defaults per contract
from pathlib import Path
import os, json, pandas as pd

CONFIG = {
    "RUN_UI": False,
    "RUN_PIPELINE": False,
    "EQUIPMENT_STATUS_PATH": "/mnt/data/equipment_status.csv",
    "EQUIPMENT_MOVES_LOG_PATH": "/mnt/data/moves_log.csv",
    "SOP_REGISTRY_PATH": "/mnt/data/sop_registry.csv",
    "QR_OUTPUT_DIR": "/mnt/data/qrs",
    "EVENT_LOG_PATH": "/mnt/data/event_log.jsonl",
}

# Ensure paths exist with CSV/dir semantics (idempotent)
from pathlib import Path
from datetime import datetime, timezone
import csv
Path(CONFIG["QR_OUTPUT_DIR"]).mkdir(parents=True, exist_ok=True)
for f, header in [
    (CONFIG["EQUIPMENT_STATUS_PATH"], ["equipment_id","location","last_seen"]),
    (CONFIG["EQUIPMENT_MOVES_LOG_PATH"], ["timestamp","equipment_id","from","to"]),
    (CONFIG["SOP_REGISTRY_PATH"], ["id","title","url"]),
]:
    f = Path(f)
    if not f.exists():
        with f.open("w", newline="") as fp:
            csv.writer(fp).writerow(header)
# Seed SOP registry if empty
import os
if os.path.getsize(CONFIG["SOP_REGISTRY_PATH"]) < 40:
    with open(CONFIG["SOP_REGISTRY_PATH"], "a", newline="") as fp:
        csv.writer(fp).writerow(["sop-0001","Universal Precautions (offline)","about:blank"])
Path(CONFIG["EVENT_LOG_PATH"]).touch(exist_ok=True)


In [32]:

# --- Demo feature toggles (safe defaults) ---
CONFIG.setdefault("RUN_MEDS", False)
CONFIG.setdefault("MED_RULES_PATH", "/mnt/data/interaction_rules.json")
CONFIG.setdefault("ALLERGIES_PATH", "/mnt/data/patient_allergies.json")
CONFIG.setdefault("RUN_SYNTH", False)
CONFIG.setdefault("SAVE_SYNTH", False)


False

In [33]:

# -- Demo meds/ICU/ML flags (non-destructive update of CONFIG) --
CONFIG.setdefault("RUN_MEDS", False)
CONFIG.setdefault("RUN_SYNTH", False)
CONFIG.setdefault("SAVE_SYNTH", False)
CONFIG.setdefault("ALLERGIES_PATH", "/mnt/data/patient_allergies.json")
CONFIG.setdefault("MED_RULES_PATH", "/mnt/data/interaction_rules.json")


'/mnt/data/interaction_rules.json'

In [34]:

# WorkflowState invariant: defined before use; exposes required methods; timers/backlogs/alerts intact
from dataclasses import dataclass, field
from typing import Any, Dict, List
import pandas as pd

@dataclass
class WorkflowState:
    role: str
    state: Dict[str, Any] = field(default_factory=dict)
    timers: Dict[str, float] = field(default_factory=dict)
    backlogs: Dict[str, List[Any]] = field(default_factory=dict)
    alerts: List[str] = field(default_factory=list)

    def touch_now(self, ts: pd.Timestamp):
        # update an example timer
        self.timers["last_touch_epoch"] = float(ts.value) / 1e9

    def feature_dict(self) -> Dict[str, Any]:
        # safe, extendable
        fd = {
            "role": self.role,
            "since_vitals_min": self.state.get("since_vitals_min", 0.0),
            "alerts_count": len(self.alerts),
        }
        # pass through any extra scalar features
        for k,v in self.state.items():
            if isinstance(v,(int,float,str)) and k not in fd:
                fd[k]=v
        return fd

    def update_state_from_event(self, event: Dict[str, Any]):
        # naive: merge event into state; track backlog
        self.state.update(event)
        self.backlogs.setdefault("events", []).append(event)

    def apply_event_log(self, events: List[Dict[str, Any]]):
        for ev in events:
            self.update_state_from_event(ev)


In [35]:

# TinyCritics uses WorkflowState.feature_dict(); cold-start safe (no transform before fit)
import numpy as np

class TinyCritics:
    def __init__(self):
        self._fitted = False

    def fit(self, states, actions, rewards):
        # No-op fit to keep cold-start safe
        self._fitted = True
        return self

    def score(self, state: WorkflowState, actions: List[Dict[str,str]]):
        fd = state.feature_dict()
        n = len(actions)
        # Deterministic, bounded scores in [0,1]
        base = 0.5
        p = np.full(n, base, dtype=float)
        bonuses = np.zeros(n, dtype=float)
        uncertainty = np.full(n, 0.1, dtype=float)
        return p, bonuses, uncertainty


In [36]:

# Inline SOP surface and optional UI without external modules
from typing import Optional, Dict, Any
import csv, os
from pathlib import Path

def refresh_sop_registry(CONFIG: dict, base_url: Optional[str]) -> Dict[str, Any]:
    """
    Offline-safe: if base_url is provided and fetch works + CSV looks valid, overwrite the file.
    Otherwise, return a summary without raising. No sidecars.
    """
    path = Path(CONFIG["SOP_REGISTRY_PATH"])
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        with path.open("w", newline="") as fp:
            csv.writer(fp).writerow(["id","title","url"])
    if base_url:
        try:
            import requests
            resp = requests.get(base_url, timeout=5)
            resp.raise_for_status()
            text = resp.text.strip()
            rows = [r.split(",") for r in text.splitlines()]
            if rows and len(rows[0])>=3:
                with path.open("w", newline="") as fp:
                    csv.writer(fp).writerows(rows)
                return {"ok": True, "rows": len(rows)-1}
        except Exception as e:
            return {"ok": False, "error": str(e)}
    return {"ok": False, "error": "No base_url or unexpected format"}

# Optional, guarded UI demo (equipment status preview uses the CSV directly)
if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W, pandas as pd
        eq_path = Path(CONFIG["EQUIPMENT_STATUS_PATH"])
        if not eq_path.exists():
            eq_path.write_text("equipment_id,location,last_seen\n")
        btn = W.Button(description="Show equipment status")
        out = W.Output()
        def _on_click(_):
            with out:
                out.clear_output()
                try:
                    df = pd.read_csv(eq_path)
                except Exception:
                    df = pd.DataFrame(columns=["equipment_id","location","last_seen"])
                display(df)
        btn.on_click(_on_click)
        display(W.VBox([btn, out]))
    except Exception as e:
        print("UI unavailable (optional):", e)


In [37]:

# Bridge: Run ML pipeline and emit ops events (guarded)
if CONFIG.get("RUN_PIPELINE"):
    from pathlib import Path
    import json
    try:
        import importlib.util
        # Run ML and emit per-row ml_risk events to /mnt/data/ml_events.jsonl
        spec = importlib.util.spec_from_file_location("ml_op_bridge", "/mnt/data/ml_op_bridge.py")
        mlb = importlib.util.module_from_spec(spec); spec.loader.exec_module(mlb)  # type: ignore
        summary = mlb.emit_ml_events()
        print("[ML] Emitted", summary["n_events"], "ml_risk events at tau=", round(summary["tau"], 4))
        # Pump into ops (lingering alerts) -> /mnt/data/event_log.jsonl
        spec2 = importlib.util.spec_from_file_location("ops_glue", "/mnt/data/ops_glue.py")
        ops = importlib.util.module_from_spec(spec2); spec2.loader.exec_module(ops)  # type: ignore
        res = ops.pump_ml_events_into_ops()
        print("[OPS] Pumped", res["n_in"], "ML events into lingering alerts")
        print("[OK] See /mnt/data/ml_events.jsonl and /mnt/data/event_log.jsonl")
    except Exception as e:
        print("[ERROR] Bridge failed:", e)
else:
    print("Deferred… set CONFIG['RUN_PIPELINE']=True to run ML→OPS bridge.")


Deferred… set CONFIG['RUN_PIPELINE']=True to run ML→OPS bridge.


In [38]:

# Optional UI: one-click ML→OPS bridge
if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W, importlib.util, json
        btn = W.Button(description="Run ML → OPS", button_style="primary")
        out = W.Output()
        def _run(_):
            with out:
                out.clear_output()
                print("Running bridge…")
                spec = importlib.util.spec_from_file_location("ml_op_bridge", "/mnt/data/ml_op_bridge.py")
                mlb = importlib.util.module_from_spec(spec); spec.loader.exec_module(mlb)  # type: ignore
                summary = mlb.emit_ml_events()
                print("[ML] Emitted", summary["n_events"], "events @ tau", round(summary["tau"], 4))
                spec2 = importlib.util.spec_from_file_location("ops_glue", "/mnt/data/ops_glue.py")
                ops = importlib.util.module_from_spec(spec2); spec2.loader.exec_module(ops)  # type: ignore
                res = ops.pump_ml_events_into_ops()
                print("[OPS] Pumped", res["n_in"], "ML events into lingering alerts")
                print("Files: /mnt/data/ml_events.jsonl, /mnt/data/event_log.jsonl")
        btn.on_click(_run)
        display(W.VBox([btn, out]))
    except Exception as e:
        print("UI unavailable:", e)
else:
    print("UI panel deferred… set CONFIG['RUN_UI']=True.")


UI panel deferred… set CONFIG['RUN_UI']=True.


In [39]:

# === Inline ML → OPS bridge (no sidecars) ===
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, Any, Optional
import json, importlib.util, os

EVENT_LOG = Path(CONFIG["EVENT_LOG_PATH"])

def _append_event(ev: Dict[str, Any]):
    EVENT_LOG.parent.mkdir(parents=True, exist_ok=True)
    EVENT_LOG.touch(exist_ok=True)
    ev = {"ts": datetime.now(timezone.utc).isoformat(), **ev}
    with EVENT_LOG.open("a") as fp:
        fp.write(json.dumps(ev, ensure_ascii=False) + "\n")

def _detect_id_col(df):
    # Prefer meta.json hinted ID columns
    meta = Path("/mnt/data/meta.json")
    if meta.exists():
        try:
            js = json.loads(meta.read_text())
            for c in js.get("validated_id_cols", []):
                if c in df.columns:
                    return c
        except Exception:
            pass
    # common fallbacks
    for c in ["Fall-ID","fall_id","PatientID","patient_id","VISIT_ID","visit_id","ID","id"]:
        if c in df.columns:
            return c
    # else: first object-like
    for c in df.columns:
        if getattr(df[c], "dtype", None) == "object":
            return c
    return df.columns[0]

def run_user_pipeline(module_path: Optional[str]=None):
    """
    Execute the user's pipeline script in /mnt/data and return (test, probs_te, tau, id_col).
    Requires the script to expose globals: test, probs_te, tau.
    """
    cand = module_path or "/mnt/data/real_data_overlap_prevalence_isotonic(1).py"
    mp = Path(cand)
    if not mp.exists():
        mp = Path("/mnt/data/real_data_overlap_prevalence_isotonic.py")
    if not mp.exists():
        raise FileNotFoundError("Pipeline script not found.")
    spec = importlib.util.spec_from_file_location("user_pipeline_mod_inline", str(mp))
    mod = importlib.util.module_from_spec(spec)
    _cwd = os.getcwd()
    try:
        os.chdir("/mnt/data")
        spec.loader.exec_module(mod)  # type: ignore
    finally:
        os.chdir(_cwd)
    missing = [k for k in ["test","probs_te","tau"] if not hasattr(mod,k)]
    if missing:
        raise RuntimeError(f"Pipeline missing globals: {missing}")
    test = getattr(mod, "test")
    probs_te = getattr(mod, "probs_te")
    tau = float(getattr(mod, "tau"))
    id_col = _detect_id_col(test)
    return test, probs_te, tau, id_col, mp.name

def ml_to_ops_emit(test, probs_te, tau: float, id_col: str, source_name: str) -> Dict[str, Any]:
    n_events = 0
    pos = 0
    for i in range(len(test)):
        pid = test.iloc[i][id_col] if id_col in test.columns else i
        prob = float(probs_te[i])
        decision = "POS" if prob >= tau else "NEG"
        _append_event({
            "type": "ml_risk",
            "id_col": id_col,
            "patient_id": pid,
            "prob_cal": round(prob, 6),
            "tau": round(tau, 6),
            "decision": decision,
            "source": source_name,
        })
        n_events += 1
        if decision == "POS":
            pos += 1
            _append_event({
                "type": "lingering_alert",
                "patient_id": pid,
                "id_col": id_col,
                "prob_cal": round(prob, 6),
                "tau": round(tau, 6),
                "reason": "ml_high_risk",
                "source": "LingeringPatientMonitor"
            })
    _append_event({
        "type": "ml_risk_summary",
        "tau": round(tau, 6),
        "n_rows": int(len(test)),
        "id_col": id_col,
        "source": source_name
    })
    return {"n_events": n_events, "n_pos": pos, "tau": tau, "id_col": id_col}

# Guarded one-shot runner
if CONFIG.get("RUN_PIPELINE"):
    try:
        test, probs_te, tau, id_col, src = run_user_pipeline()
        res = ml_to_ops_emit(test, probs_te, tau, id_col, src)
        print("[OK] Emitted:", res)
        print("→ EVENT_LOG:", CONFIG["EVENT_LOG_PATH"])
    except Exception as e:
        print("[ERROR]", e)
else:
    print("Deferred… set CONFIG['RUN_PIPELINE']=True to run ML→OPS inline.")


Deferred… set CONFIG['RUN_PIPELINE']=True to run ML→OPS inline.


In [40]:

# Inline UI trigger (no sidecars)
if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W
        btn = W.Button(description="Run ML → OPS (inline)", button_style="primary")
        out = W.Output()
        def _go(_):
            with out:
                out.clear_output()
                print("Running…")
                try:
                    test, probs_te, tau, id_col, src = run_user_pipeline()
                    res = ml_to_ops_emit(test, probs_te, tau, id_col, src)
                    print("[OK] Emitted:", res)
                except Exception as e:
                    print("[ERROR]", e)
        btn.on_click(_go)
        display(W.VBox([btn, out]))
    except Exception as e:
        print("UI unavailable:", e)
else:
    print("UI panel deferred… set CONFIG['RUN_UI']=True.")


UI panel deferred… set CONFIG['RUN_UI']=True.


In [41]:
# ICU availability panel — replaces "actionable vs blocked" with explicit next-bed ETAs.
# No sidecars; all inline; guarded by RUN_UI.
from pathlib import Path
from datetime import datetime, timezone
import json

# Pre-seeded UKE ICUs (public info): names + capacities
UKE_UNITS = [
    {"name": "1A Neurochirurgische Intensivstation", "capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1B Neurologische Intensivstation",    "capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1C Interdisziplinäre Intensivstation","capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1D Interdisziplinäre Intensivstation","capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1E Interdisziplinäre Intensivstation","capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1F Operative Intensivstation",        "capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1G Internistische Intensivstation",   "capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "H1b Kardiologische Intensivstation",  "capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "H1b Kardiochirurgische Intensivstation","capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "H2b Intensivstation Gefäß- und Herzmedizin", "capacity": 8, "occupied": 8, "discharge_eta_minutes": []},
]

def _format_eta(mins: int) -> str:
    if mins is None: return "unknown"
    if mins <= 0: return "now"
    h, m = divmod(int(mins), 60)
    return f"{m} min" if h == 0 else (f"{h} hr" if m == 0 else f"{h} hr {m} min")

def _load_icu_status(path="/mnt/data/icu_status.json"):
    p = Path(path)
    if p.exists():
        try:
            js = json.loads(p.read_text())
            if isinstance(js, dict) and js.get("units"):
                return js
        except Exception:
            pass
    # default to UKE units when nothing saved
    return {"units": list(UKE_UNITS), "timestamp": datetime.now(timezone.utc).isoformat()}

def _save_icu_status(js, path="/mnt/data/icu_status.json"):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(js, ensure_ascii=False, indent=2))

def _compute_next_bed_eta(unit):
    cap = int(unit.get("capacity", 0) or 0)
    occ = int(unit.get("occupied", 0) or 0)
    etas = [int(x) for x in (unit.get("discharge_eta_minutes") or []) if str(x).strip().isdigit()]
    if occ < cap: return 0
    return min(etas) if etas else None

if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W
        import pandas as pd

        state = _load_icu_status()
        units = state["units"]

        # UI widgets
        dd = W.Dropdown(options=[u.get("name","(unnamed)") for u in units] or ["(add a unit)"], description="Unit")
        name = W.Text(description="Name", placeholder="ICU-North")
        cap  = W.IntText(description="Capacity", value=12)
        occ  = W.IntText(description="Occupied", value=12)
        eta  = W.Text(description="ETAs (min)", placeholder="e.g. 30, 90, 180")

        add_btn = W.Button(description="Add/Update unit")
        calc_btn = W.Button(description="Estimate ETAs")
        save_btn = W.Button(description="Save status")
        out = W.Output()

        def _refresh_dropdown():
            dd.options = [u.get("name","(unnamed)") for u in units] or ["(add a unit)"]

        def _load_into_form(idx=0):
            if not units:
                name.value=""; cap.value=12; occ.value=12; eta.value=""; return
            u = units[idx]
            name.value = str(u.get("name",""))
            cap.value = int(u.get("capacity", 0) or 0)
            occ.value = int(u.get("occupied", 0) or 0)
            seq = u.get("discharge_eta_minutes") or []
            eta.value = ", ".join(str(int(x)) for x in seq)

        def _parse_eta(txt: str):
            out = []
            for chunk in txt.split(","):
                chunk = chunk.strip()
                if chunk:
                    try: out.append(int(float(chunk)))
                    except: pass
            return out

        def on_dd_change(change):
            if change["name"]=="value" and units:
                _load_into_form(dd.options.index(change["new"]))
        dd.observe(on_dd_change)

        def on_add(_):
            # no 'nonlocal' needed: we mutate the existing list
            u = {
                "name": name.value.strip() or f"ICU-{len(units)+1}",
                "capacity": int(cap.value or 0),
                "occupied": int(occ.value or 0),
                "discharge_eta_minutes": _parse_eta(eta.value),
            }
            names = [x.get("name","") for x in units]
            if u["name"] in names:
                units[names.index(u["name"])] = u
            else:
                units.append(u)
            _refresh_dropdown()
            dd.value = u["name"]
            with out:
                print(f"Saved unit '{u['name']}'")

        def on_calc(_):
            rows = []
            for u in units:
                eta_min = _compute_next_bed_eta(u)
                rows.append({
                    "ICU": u.get("name",""),
                    "capacity": int(u.get("capacity",0) or 0),
                    "occupied": int(u.get("occupied",0) or 0),
                    "next_bed_in": _format_eta(eta_min),
                })
            df = pd.DataFrame(rows) if rows else pd.DataFrame(columns=["ICU","capacity","occupied","next_bed_in"])
            with out:
                out.clear_output()
                if df.empty:
                    print("No units yet. Add a unit above.")
                else:
                    display(df.style.hide(axis='index'))
                    # Natural-language earliest
                    mins = [(r["ICU"], _compute_next_bed_eta(u)) for r,u in zip(rows, units)]
                    mins = [(n,m) for n,m in mins if m is not None]
                    if mins:
                        name_min, m = sorted(mins, key=lambda x: x[1])[0]
                        print(f"\nNext bed available on {name_min} in {_format_eta(m)}")
                    else:
                        print("\nNext bed availability: unknown (provide ETAs or reduce occupied < capacity).")

        def on_save(_):
            state["units"] = units
            state["timestamp"] = datetime.now(timezone.utc).isoformat()
            _save_icu_status(state)
            with out:
                print("Saved to /mnt/data/icu_status.json")

        add_btn.on_click(on_add)
        calc_btn.on_click(on_calc)
        save_btn.on_click(on_save)

        # Initial load
        _refresh_dropdown()
        if units:
            dd.value = dd.options[0]
            _load_into_form(0)

        display(W.VBox([
            W.HTML("<b>ICU next-bed availability</b>"),
            dd,
            W.HBox([name, cap, occ]),
            eta,
            W.HBox([add_btn, calc_btn, save_btn]),
            out
        ]))

    except Exception as e:
        print("ICU UI unavailable:", e)
else:
    print("ICU UI deferred… set CONFIG['RUN_UI']=True.")


ICU UI deferred… set CONFIG['RUN_UI']=True.


In [42]:

# === Realistic synthetic data pipeline (inline, no sidecars) ===
# Guard: RUN_SYNTH controls generation + training + event emission
from __future__ import annotations
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd, numpy as np, json
from typing import Dict, Any, Optional
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_curve, precision_recall_fscore_support, roc_auc_score

BASE = Path("/mnt/data")
EVENT_LOG = Path(CONFIG["EVENT_LOG_PATH"])

def _append_event(ev: Dict[str, Any]):
    EVENT_LOG.parent.mkdir(parents=True, exist_ok=True)
    EVENT_LOG.touch(exist_ok=True)
    ev = {"ts": datetime.now(timezone.utc).isoformat(), **ev}
    with EVENT_LOG.open("a") as fp:
        fp.write(json.dumps(ev, ensure_ascii=False) + "\n")

def _detect_label_and_id(train: pd.DataFrame, meta_path=BASE/"meta.json"):
    meta = {}
    if Path(meta_path).exists():
        try:
            meta = json.loads(Path(meta_path).read_text())
        except Exception:
            meta = {}
    label_col = meta.get("label_col")
    if not label_col:
        for c in train.columns:
            if pd.api.types.is_numeric_dtype(train[c]):
                u = set(pd.unique(train[c].dropna()))
                if u.issubset({0,1}):
                    label_col = c; break
    if not label_col:
        raise RuntimeError("Could not detect label column")
    id_col = None
    for c in meta.get("validated_id_cols", []):
        if c in train.columns: id_col = c; break
    for c in ["Fall-ID","fall_id","PatientID","patient_id","VISIT_ID","visit_id","ID","id"]:
        if id_col is None and c in train.columns: id_col = c; break
    if id_col is None: id_col = train.columns[0]
    return label_col, id_col

def _split_features(df: pd.DataFrame, id_col: str, label_col: str):
    num = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    cat = [c for c in df.columns if c not in num]
    drop = set([id_col, label_col])
    num = [c for c in num if c not in drop]
    cat = [c for c in cat if c not in drop]
    low_card = [c for c in cat if df[c].nunique(dropna=True) <= 30]
    return num, low_card

def _numeric_params(s: pd.Series):
    s_nonnull = s.dropna()
    if len(s_nonnull)==0:
        return {"mean":0.0,"std":1.0,"lo":0.0,"hi":1.0,"missing":1.0}
    mean = float(s_nonnull.mean()); std = float(s_nonnull.std(ddof=0) or 1.0)
    lo = float(np.percentile(s_nonnull, 1)); hi = float(np.percentile(s_nonnull, 99))
    missing = float(s.isna().mean())
    return {"mean":mean,"std":std,"lo":lo,"hi":hi,"missing":missing}

def _sample_numeric(n, p):
    x = np.random.normal(p["mean"], p["std"], size=n)
    x = np.clip(x, p["lo"], p["hi"])
    if p["missing"]>0:
        m = np.random.rand(n) < p["missing"]
        x = x.astype("float"); x[m] = np.nan
    return x

def _categorical_params(s: pd.Series):
    missing = float(s.isna().mean())
    counts = s.dropna().value_counts()
    if counts.empty:
        return {"cats":["UNK"],"probs":[1.0],"missing":1.0}
    cats = counts.index.tolist(); probs = (counts/counts.sum()).values.tolist()
    return {"cats":cats,"probs":probs,"missing":missing}

def _sample_categorical(n, p):
    base = np.random.choice(p["cats"], size=n, p=p["probs"])
    if p["missing"]>0:
        m = np.random.rand(n) < p["missing"]
        base = base.astype("object"); base[m] = None
    return base

def _prep_X(df: pd.DataFrame, num_cols, cat_cols, all_cols=None):
    X = pd.get_dummies(df[num_cols + cat_cols], columns=cat_cols, dummy_na=True)
    if all_cols is not None:
        for c in all_cols:
            if c not in X.columns: X[c] = 0
        X = X[all_cols]
    return X

def run_synth_pipeline(save_csv: bool=False):
    # Load real splits
    train = pd.read_csv(BASE/"train_DE_full.csv")
    val   = pd.read_csv(BASE/"val_DE_full.csv")
    test  = pd.read_csv(BASE/"test_DE_full.csv")

    label_col, id_col = _detect_label_and_id(train)
    num_cols, cat_cols = _split_features(train, id_col, label_col)

    # Fit a real model on real features (to induce structure for labels)
    Xr = _prep_X(train, num_cols, cat_cols); Xv = _prep_X(val, num_cols, cat_cols); Xt = _prep_X(test, num_cols, cat_cols)
    all_cols = sorted(set(Xr.columns) | set(Xv.columns) | set(Xt.columns))
    Xr = _prep_X(train, num_cols, cat_cols, all_cols); y_real = train[label_col].astype(int).values

    base_pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc", StandardScaler(with_mean=False)),
                          ("lr", LogisticRegression(max_iter=1000))])
    base_pipe.fit(Xr, y_real)
    target_prev = float(np.mean(y_real))

    # Params from real TRAIN for synthesis
    num_param_map = {c: _numeric_params(train[c]) for c in num_cols}
    cat_param_map = {c: _categorical_params(train[c]) for c in cat_cols}

    def _make_ids(n): 
        import uuid
        return [f"SYN-{uuid.uuid4().hex[:10]}" for _ in range(n)]

    def synth_df(n_rows: int) -> pd.DataFrame:
        data = {id_col: _make_ids(n_rows)}
        for c in cat_cols: data[c] = _sample_categorical(n_rows, cat_param_map[c])
        for c in num_cols: data[c] = _sample_numeric(n_rows, num_param_map[c])
        return pd.DataFrame(data)

    n_tr, n_va, n_te = len(train), len(val), len(test)
    syn_tr = synth_df(n_tr); syn_va = synth_df(n_va); syn_te = synth_df(n_te)

    def label_from_base(df: pd.DataFrame) -> np.ndarray:
        X = _prep_X(df, num_cols, cat_cols, all_cols)
        scores = base_pipe.predict_proba(X)[:,1]
        thr = np.quantile(scores, 1-target_prev) if 0<target_prev<1 else 0.5
        return (scores >= thr).astype(int)

    for df in (syn_tr, syn_va, syn_te):
        df[label_col] = label_from_base(df)

    # Reorder
    def _reorder(df): 
        cols = [id_col, label_col] + [c for c in df.columns if c not in [id_col, label_col]]
        return df[cols]
    syn_tr, syn_va, syn_te = map(_reorder, (syn_tr, syn_va, syn_te))

    # Train calibrated model on SYNTH
    X_tr = _prep_X(syn_tr, num_cols, cat_cols); y_tr = syn_tr[label_col].astype(int).values
    X_va = _prep_X(syn_va, num_cols, cat_cols); y_va = syn_va[label_col].astype(int).values
    X_te = _prep_X(syn_te, num_cols, cat_cols); y_te = syn_te[label_col].astype(int).values
    all_synth_cols = sorted(set(X_tr.columns) | set(X_va.columns) | set(X_te.columns))
    X_tr = _prep_X(syn_tr, num_cols, cat_cols, all_synth_cols)
    X_va = _prep_X(syn_va, num_cols, cat_cols, all_synth_cols)
    X_te = _prep_X(syn_te, num_cols, cat_cols, all_synth_cols)

    base = Pipeline([("imp", SimpleImputer(strategy="median")),
                     ("sc", StandardScaler(with_mean=False)),
                     ("lr", LogisticRegression(max_iter=1000))]).fit(X_tr, y_tr)
    cal = CalibratedClassifierCV(base, method="isotonic", cv="prefit").fit(X_va, y_va)

    probs_va = cal.predict_proba(X_va)[:,1]
    fpr, tpr, thr = roc_curve(y_va, probs_va)
    RECALL_FLOOR = 0.85
    meet = np.where(tpr >= RECALL_FLOOR)[0]
    if len(meet)>0:
        tau = float(thr[meet[0]])
    else:
        best_f1, best_t = -1, 0.5
        for t in np.linspace(0,1,201):
            yb = (probs_va >= t).astype(int)
            _, _, f1, _ = precision_recall_fscore_support(y_va, yb, average="binary", zero_division=0)
            if f1 > best_f1: best_f1, best_t = f1, t
        tau = float(best_t)

    probs_te = cal.predict_proba(X_te)[:,1]
    auc_te = float(roc_auc_score(y_te, probs_te))

    # Emit events (ml_risk_synth + lingering_alert_synth + summary)
    n_events = 0; n_pos = 0
    for i in range(len(syn_te)):
        pid = syn_te.iloc[i][id_col]
        p = float(probs_te[i])
        decision = "POS" if p >= tau else "NEG"
        ev = {"type":"ml_risk_synth","patient_id": pid,"id_col": id_col,
              "prob_cal": round(p,6), "tau": round(tau,6),"decision": decision,
              "source":"synth_pipeline"}
        _append_event(ev)
        n_events += 1
        if decision=="POS":
            n_pos += 1
            _append_event({"type":"lingering_alert_synth","patient_id": pid,"id_col": id_col,
                           "prob_cal": round(p,6), "tau": round(tau,6),
                           "reason":"ml_high_risk_synth","source":"LingeringPatientMonitor"})

    _append_event({"type":"ml_risk_summary_synth","tau": round(tau,6),"n_rows": int(len(syn_te)),
                   "id_col": id_col, "auc_te": round(auc_te,6), "source":"synth_pipeline"})

    # Optionally save CSVs
    if CONFIG.get("SAVE_SYNTH"):
        syn_tr.to_csv(BASE/"synth_train.csv", index=False)
        syn_va.to_csv(BASE/"synth_val.csv", index=False)
        syn_te.to_csv(BASE/"synth_test.csv", index=False)

    return {"n_events": n_events, "n_pos": n_pos, "tau": tau, "auc_te": auc_te, "id_col": id_col}

if CONFIG.get("RUN_SYNTH"):
    try:
        res = run_synth_pipeline(save_csv=bool(CONFIG.get("SAVE_SYNTH", False)))
        print("[SYNTH OK]", res)
        print("Events in:", CONFIG["EVENT_LOG_PATH"])
    except Exception as e:
        print("[SYNTH ERROR]", e)
else:
    print("Deferred… set CONFIG['RUN_SYNTH']=True to generate + run synthetic pipeline.")


Deferred… set CONFIG['RUN_SYNTH']=True to generate + run synthetic pipeline.


In [43]:

# Extend CONFIG for meds integration (idempotent)
CONFIG.update({
    "MED_RULES_PATH": "/mnt/data/interaction_rules.json",
    "ALLERGIES_PATH": "/mnt/data/patient_allergies.json",
    "RUN_MEDS": False,   # gate for meds OCR/parse pipeline
})
from pathlib import Path, PurePosixPath
# touch files if absent (empty defaults)
for p in [CONFIG["MED_RULES_PATH"], CONFIG["ALLERGIES_PATH"]]:
    pp = Path(p)
    if not pp.exists():
        try:
            # Minimal demo defaults
            if p.endswith("interaction_rules.json"):
                pp.write_text(json.dumps({
                    "pair_rules":[
                        # DEMO ONLY: not for clinical use. Replace with licensed source in production.
                        {"lhs":"sildenafil","rhs":"nitroglycerin","severity":"contraindicated","note":"Do not co-administer (demo)."},
                        {"lhs":"warfarin","rhs":"ibuprofen","severity":"major","note":"Bleeding risk (demo)."},
                        {"lhs":"ramipril","rhs":"spironolactone","severity":"major","note":"Hyperkalemia risk (demo)."},
                        {"lhs":"sertraline","rhs":"tramadol","severity":"moderate","note":"Serotonergic effects (demo)."},
                        {"lhs":"metformin","rhs":"iv contrast","severity":"moderate","note":"Renal function dependent (demo)."}
                    ],
                    "dup_atc_level": 5  # consider same 5th-level ATC a duplicate (demo)
                }, ensure_ascii=False, indent=2))
            elif p.endswith("patient_allergies.json"):
                pp.write_text(json.dumps({
                    # Example patient allergies (empty by default). Names free-text.
                    "allergies": []
                }, ensure_ascii=False, indent=2))
        except Exception:
            pass


In [44]:

# === Medication Plan Import: OCR + Parse + Interaction/Allergy checks (inline, no sidecars) ===
from __future__ import annotations
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from datetime import datetime, timezone
import json, re

EVENT_LOG = Path(CONFIG["EVENT_LOG_PATH"])

def _append_event(ev: Dict[str, Any]):
    EVENT_LOG.parent.mkdir(parents=True, exist_ok=True)
    EVENT_LOG.touch(exist_ok=True)
    ev = {"ts": datetime.now(timezone.utc).isoformat(), **ev}
    with EVENT_LOG.open("a") as fp:
        fp.write(json.dumps(ev, ensure_ascii=False) + "\n")

# --- OCR (best-effort, optional) ---
def try_ocr(path: str) -> Optional[str]:
    """
    Try to OCR a BMP scan from an image/PDF.
    Returns UTF-8 text or None if OCR is unavailable.
    """
    try:
        from PIL import Image
        import pytesseract
        p = Path(path)
        if not p.exists():
            return None
        if p.suffix.lower() in {".png",".jpg",".jpeg",".tif",".tiff",".bmp"}:
            img = Image.open(p)
            txt = pytesseract.image_to_string(img, lang="deu")
            return txt
        elif p.suffix.lower() in {".pdf"}:
            # Lightweight PDF approach: try to use pdf2image if available, else None
            try:
                from pdf2image import convert_from_path
                pages = convert_from_path(str(p))
                chunks = []
                for im in pages[:5]:  # limit first 5 pages
                    chunks.append(pytesseract.image_to_string(im, lang="deu"))
                return "\n\n".join(chunks)
            except Exception:
                return None
        else:
            return None
    except Exception:
        return None

# --- BMP parser (heuristic) ---
_ATC_MAP = {
    # DEMO, extend as needed. Not for clinical use.
    "amoxicillin":"J01CA04","ibuprofen":"M01AE01","metoprolol":"C07AB02","ramipril":"C09AA05",
    "simvastatin":"C10AA01","acetylsalicylsäure":"B01AC06","aspirin":"B01AC06","warfarin":"B01AA03",
    "metformin":"A10BA02","insulin glargin":"A10AE04","insulin glargine":"A10AE04",
    "pantoprazol":"A02BC02","pantoprazole":"A02BC02","sertralin":"N06AB06","sertraline":"N06AB06",
    "spironolacton":"C03DA01","spironolactone":"C03DA01","sildenafil":"G04BE03",
    "nitroglycerin":"C01DA02","isosorbidmononitrat":"C01DA14","tramadol":"N02AX02"
}
_UNIT_PAT = r"(mg|mcg|µg|g|IE|I\.E\.|ml|mL)"
_DOSE_PAT = r"(?:(\d+(?:[.,]\d+)?)\s*"+_UNIT_PAT+r")"
_FREQ_PAT = r"\b(\d-\d-\d(?:-\d)?)\b|\b(morgens|mittags|abends|nachts)(?:[,/ ]+(morgens|mittags|abends|nachts))*\b"
_ROUTE_PAT = r"\b(p\.o\.|oral|i\.v\.|i\.m\.|s\.c\.|iv|im|sc|po|SL)\b"
_PRN_PAT = r"\bPRN\b|\bbei Bedarf\b"

def normalize(s: str) -> str:
    return re.sub(r"\s+", " ", s.strip())

def parse_med_line(line: str) -> Optional[Dict[str, Any]]:
    s = line.strip()
    if not s: return None
    # quick reject of headers
    if re.search(r"(?i)medikationsplan|bundeseinheitlich|name des patienten|geburtsdatum|arzt|datum", s):
        return None
    # Split name + rest
    m = re.search(_DOSE_PAT, s, flags=re.I)
    name_part = s
    strength_val = None; strength_unit = None
    if m:
        name_part = s[:m.start()].strip(" -•:")
        strength_val = m.group(1).replace(",",".") if m.group(1) else None
        strength_unit = m.group(2)
    # freq
    freq = None
    mf = re.search(_FREQ_PAT, s, flags=re.I)
    if mf:
        freq = mf.group(0)
    # route
    mr = re.search(_ROUTE_PAT, s, flags=re.I)
    route = mr.group(0) if mr else None
    # prn
    prn = bool(re.search(_PRN_PAT, s, flags=re.I))

    name_clean = normalize(name_part.lower())
    if not name_clean:
        return None
    # take first token as candidate; remove extraneous punctuation
    base = re.sub(r"[^\wäöüß\- ]", "", name_clean).strip()
    # map to ATC where possible
    atc = None
    for k,v in _ATC_MAP.items():
        if k in base:
            atc = v; break
    return {
        "raw": s,
        "name": base,
        "strength": float(strength_val) if strength_val else None,
        "unit": strength_unit,
        "freq": freq,
        "route": route,
        "prn": prn,
        "atc": atc
    }

def parse_med_text(text: str) -> List[Dict[str, Any]]:
    meds = []
    # split on lines/bullets/semicolons
    for raw in re.split(r"[\n;\u2022]+", text):
        m = parse_med_line(raw)
        if m: meds.append(m)
    # de-dup by name+strength+unit
    uniq = {}
    for m in meds:
        key = (m["name"], m.get("strength"), m.get("unit"))
        if key not in uniq:
            uniq[key] = m
    return list(uniq.values())

# --- Interaction & allergy checking (pluggable rules) ---
def load_rules(path: str) -> Dict[str, Any]:
    try:
        js = json.loads(Path(path).read_text())
        return js if isinstance(js, dict) else {}
    except Exception:
        return {}

def load_allergies(path: str) -> Dict[str, Any]:
    try:
        js = json.loads(Path(path).read_text())
        if isinstance(js, dict) and "allergies" in js:
            return js
    except Exception:
        pass
    return {"allergies": []}

def atc_prefix(code: Optional[str], level=4) -> Optional[str]:
    if not code: return None
    code = code.strip().upper()
    # ATC format: A10BA02 -> levels 1:A, 2:A10, 3:A10B, 4:A10BA, 5:A10BA02
    if level==5: return code
    if level==4: return code[:5]
    if level==3: return code[:4]
    if level==2: return code[:3]
    if level==1: return code[:1]
    return code

def check_interactions(meds: List[Dict[str,Any]], rules: Dict[str,Any]) -> List[Dict[str,Any]]:
    warns = []
    # duplicate therapy via ATC 5th level
    dup_level = int(rules.get("dup_atc_level", 5))
    seen = {}
    for m in meds:
        pref = atc_prefix(m.get("atc"), dup_level)
        if pref:
            seen.setdefault(pref, []).append(m)
    for pref, group in seen.items():
        if len(group) > 1:
            warns.append({"type":"dup_therapy","atc_prefix":pref,"agents":[g.get("name") for g in group],
                          "severity":"info","note":f"Multiple agents in same ATC level {dup_level} (demo)"})
    # pairwise rules
    prules = rules.get("pair_rules", [])
    names = [m.get("name","") for m in meds]
    lower_names = [n.lower() for n in names]
    for r in prules:
        lhs = r.get("lhs","").lower()
        rhs = r.get("rhs","").lower()
        # match by name substring OR by ATC code exact if provided
        found_l = any(lhs in n for n in lower_names)
        found_r = any(rhs in n for n in lower_names)
        if found_l and found_r:
            warns.append({"type":"interaction_pair","lhs":lhs,"rhs":rhs,
                          "severity":r.get("severity","unknown"),"note":r.get("note","")})
    return warns

def check_allergies(meds: List[Dict[str,Any]], allergen_list: List[str]) -> List[Dict[str,Any]]:
    warns = []
    lowers = [a.lower() for a in allergen_list]
    for m in meds:
        nm = m.get("name","").lower()
        atc = (m.get("atc") or "").upper()
        if any(a in nm for a in lowers):
            warns.append({"type":"allergy_match","agent":m.get("name"),"severity":"major","note":"Listed allergy matches med name (string match)."})
        # soft cross-reactivity heuristic: same ATC first letter (very rough; DEMO ONLY)
        if atc and any(atc.startswith(a.upper()[:1]) for a in lowers if len(a)>=1):
            warns.append({"type":"allergy_crossreactivity_heuristic","agent":m.get("name"),
                          "severity":"info","note":"Heuristic ATC-class proximity; verify clinically."})
    return warns

def meds_pipeline_from_text(text: str, patient_id: Optional[str]=None) -> Dict[str,Any]:
    meds = parse_med_text(text)
    rules = load_rules(CONFIG["MED_RULES_PATH"])
    allergies = load_allergies(CONFIG["ALLERGIES_PATH"]).get("allergies", [])
    inter = check_interactions(meds, rules)
    alrx = check_allergies(meds, allergies)
    # Emit events
    _append_event({"type":"med_plan_import","n_meds": len(meds),"patient_id": patient_id})
    for m in meds:
        _append_event({"type":"med_entry","patient_id":patient_id, **m})
    for w in inter:
        _append_event({"type":"med_interaction_warning","patient_id":patient_id, **w})
    for w in alrx:
        _append_event({"type":"med_allergy_warning","patient_id":patient_id, **w})
    return {"meds": meds, "interactions": inter, "allergy_warnings": alrx}

def meds_pipeline_from_scan(path: str, patient_id: Optional[str]=None) -> Dict[str,Any]:
    txt = try_ocr(path)
    if not txt:
        return {"error":"OCR unavailable or failed; paste text instead."}
    return meds_pipeline_from_text(txt, patient_id=patient_id)

if CONFIG.get("RUN_MEDS"):
    print("Medication import pipeline is enabled. Use the UI panel (if RUN_UI) or call meds_pipeline_* functions.")
else:
    print("Deferred… set CONFIG['RUN_MEDS']=True to enable medication import pipeline.")


Deferred… set CONFIG['RUN_MEDS']=True to enable medication import pipeline.


In [45]:

# UI: Medication Plan Import (scan or paste), Interaction/Allergy checks — inline, no sidecars
if CONFIG.get("RUN_UI") and CONFIG.get("RUN_MEDS"):
    try:
        import ipywidgets as W
        import pandas as pd
        from IPython.display import display

        patient = W.Text(value="", description="Patient ID")
        path_in = W.Text(value="", placeholder="/mnt/data/scan.png or .pdf", description="Scan path")
        ocr_btn = W.Button(description="Run OCR from scan")
        paste = W.Textarea(value="", placeholder="Paste BMP text here…", description="Text", layout=W.Layout(width="90%", height="120px"))
        parse_btn = W.Button(description="Parse & Check")
        out = W.Output()

        def _to_df(items: list[dict], cols=None):
            if not items: return pd.DataFrame()
            df = pd.DataFrame(items)
            return df if cols is None else df.reindex(columns=cols, fill_value="")

        def on_ocr(_):
            with out:
                out.clear_output()
                res = meds_pipeline_from_scan(path_in.value.strip(), patient_id=patient.value.strip() or None)
                if "error" in res:
                    print("OCR error:", res["error"])
                    return
                mdf = _to_df(res["meds"], cols=["name","strength","unit","freq","route","prn","atc","raw"])
                idf = _to_df(res["interactions"])
                adf = _to_df(res["allergy_warnings"])
                print("Parsed meds:"); display(mdf)
                if not idf.empty: print("\nInteractions:"); display(idf)
                if not adf.empty: print("\nAllergy warnings:"); display(adf)

        def on_parse(_):
            with out:
                out.clear_output()
                res = meds_pipeline_from_text(paste.value, patient_id=patient.value.strip() or None)
                mdf = _to_df(res["meds"], cols=["name","strength","unit","freq","route","prn","atc","raw"])
                idf = _to_df(res["interactions"])
                adf = _to_df(res["allergy_warnings"])
                print("Parsed meds:"); display(mdf)
                if not idf.empty: print("\nInteractions:"); display(idf)
                if not adf.empty: print("\nAllergy warnings:"); display(adf)

        ocr_btn.on_click(on_ocr)
        parse_btn.on_click(on_parse)

        display(W.VBox([
            W.HTML("<b>Medication Plan Import (scan or paste)</b>"),
            patient,
            W.HBox([path_in, ocr_btn]),
            paste,
            parse_btn,
            out
        ]))
    except Exception as e:
        print("Meds UI unavailable:", e)
else:
    print("Meds UI deferred… set CONFIG['RUN_UI']=True and CONFIG['RUN_MEDS']=True.")


Meds UI deferred… set CONFIG['RUN_UI']=True and CONFIG['RUN_MEDS']=True.


In [46]:

# === BMP (Bundeseinheitlicher Medikationsplan) scanner + importer (inline, no sidecars) ===
# Purpose: decode DataMatrix barcode from the printed BMP, parse XML "Carriersegment", extract meds,
#          run allergy cross-reactivity + interaction heuristics, and stage import payloads.
# Guards: RUN_UI controls widget panel; no background jobs.
from __future__ import annotations
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, Any, List, Optional, Tuple
import json, io

# Soft deps
def _try_imports():
    dm = None
    try:
        from pylibdmtx.pylibdmtx import decode as dm_decode  # DataMatrix
        from PIL import Image
        dm = (dm_decode, Image)
    except Exception:
        dm = None
    return dm

def _decode_datamatrix_from_image(path: str) -> Optional[str]:
    dm = _try_imports()
    if dm is None:
        return None
    dm_decode, Image = dm
    try:
        img = Image.open(path)
        res = dm_decode(img)
        if res:
            # Choose the longest decoded payload
            payload = max(res, key=lambda r: len(r.data)).data
            try:
                return payload.decode("iso-8859-1", errors="ignore")
            except Exception:
                return payload.decode("utf-8", errors="ignore")
    except Exception:
        return None
    return None

# Minimal XML parser for the BMP "Carriersegment" (per KBV spec: XML, root <MP ...>)
def _parse_bmp_xml(xml_text: str) -> Dict[str, Any]:
    import xml.etree.ElementTree as ET
    meds = []
    meta = {}
    try:
        root = ET.fromstring(xml_text.strip())
    except Exception as e:
        return {"ok": False, "error": f"XML parse failed: {e}"}
    # Root attributes include version v="022" etc.
    meta["root_tag"] = root.tag
    meta["version"] = root.attrib.get("v")
    # Heuristic: medication entries appear as child elements (e.g., <M/> or <A/> etc. depending on schema);
    # We'll scan for elements carrying PZN ("P" or "PZN"), substance ("WST"), strength, dose instructions.
    for el in root.iter():
        tag = el.tag
        at = el.attrib or {}
        # Look for attributes commonly used in samples
        pzn = at.get("P") or at.get("PZN") or None
        name = at.get("HN") or at.get("NAME") or at.get("TXT") or None  # Handelsname / text
        wst  = at.get("WST") or at.get("WS") or None  # Wirkstoff(e)
        strg = at.get("ST") or at.get("STR") or at.get("STAERKE") or None
        dose = at.get("DA") or at.get("DOS") or at.get("DOSE") or None
        freq = at.get("FREQ") or at.get("FRQ") or None
        route = at.get("ROUTE") or at.get("ANW") or None
        form = at.get("DAR") or at.get("DARZ") or None  # Darreichungsform (IFA code)
        if any([pzn, name, wst, strg, dose]):
            meds.append({
                "pzn": pzn,
                "name": name,
                "substance": wst,
                "strength": strg,
                "dose": dose,
                "freq": freq,
                "route": route,
                "form": form,
                "_tag": tag,
                "_raw": at
            })
    # Deduplicate meds by pzn + name + substance
    seen = set()
    uniq = []
    for m in meds:
        key = (m.get("pzn"), m.get("name"), m.get("substance"), m.get("strength"), m.get("dose"))
        if key not in seen:
            seen.add(key); uniq.append(m)
    return {"ok": True, "meta": meta, "meds": uniq}

# Allergy + interaction heuristics (non-decisional, warnings only)
ALLERGY_GROUPS = {
    "penicillin": {"aliases": ["penicillin", "amoxicillin", "ampicillin", "oxacillin", "piperacillin", "flucloxacillin"]},
    "cephalosporin": {"aliases": ["cephalosporin", "cefaclor", "cefazolin", "cefalexin", "cefuroxim", "ceftriaxon", "ceftazidim", "cefpodoxim"]},
    "sulfonamide_abx": {"aliases": ["sulfamethoxazole", "trimethoprim-sulfamethoxazole", "co-trimoxazole"]},
    "nsaid": {"aliases": ["ibuprofen","diclofenac","naproxen","aspirin","ketorolac","indometacin","piroxicam"]},
    "opioid": {"aliases": ["morphin","oxycodon","fentanyl","hydromorphon","tramadol","codein","tilidin"]},
}

CROSS_REACTIVITY_RULES = [
    # Rule format: (allergy_group, suspect_group, note)
    ("penicillin", "cephalosporin", "Beta-Lactam cross-reactivity possible; clinical relevance depends on generation/side-chain; review carefully."),
    ("nsaid", "nsaid", "NSAID hypersensitivity often cross-reactive across nonselective NSAIDs; consider COX-2 selection if appropriate."),
    ("sulfonamide_abx", "sulfonamide_abx", "Avoid sulfonamide antibiotics if sulfonamide antibiotic allergy is present."),
]

# Interaction rules (coarse). Pattern match on substance or name substrings (casefold); non-decisional warnings.
INTERACTION_RULES = [
    (["warfarin","phenprocoumon","acenocoumarol"], ["nsaid","ibuprofen","diclofenac","naproxen","aspirin"],
     "Anticoagulant + NSAID → bleeding risk; consider gastroprotection/alternatives; monitor INR if VKA."),
    (["warfarin","phenprocoumon","acenocoumarol"], ["macrolid","erythromycin","clarithromycin","azithromycin","fluoroquinolon","ciprofloxacin","levofloxacin"],
     "VKA + certain antibiotics → ↑INR/bleeding; monitor closely."),
    (["simvastatin","lovastatin"], ["clarithromycin","erythromycin","itraconazol","ketoconazol","posaconazol","voriconazol"],
     "Statin (CYP3A4) + strong inhibitor → ↑rhabdomyolysis risk; consider hold/switch."),
    (["ssri","citalopram","escitalopram","sertralin","fluoxetin","paroxetin","duloxetin","venlafaxin"], ["maoi","triptan","linezolid"],
     "Serotonergic combo → serotonin syndrome risk; avoid/monitor."),
    (["ace","ramipril","enalapril","lisinopril","sartan","valsartan","losartan"], ["spironolacton","eplerenon","amilorid","triamteren","kalium"],
     "RAAS blocker + K-sparing / potassium → hyperkalaemia risk; check K+/renal function."),
    (["doac","apixaban","rivaroxaban","edoxaban","dabigatran"], ["dual antiplatelet","clopidogrel","prasugrel","ticagrelor","aspirin"],
     "Anticoagulant + antiplatelet(s) → bleeding risk; verify indication and duration."),
    (["qt","amiodaron","sotalol","haloperidol","ciprofloxacin","levofloxacin","clarithromycin","erythromycin","citalopram","escitalopram"], 
     ["qt","amiodaron","sotalol","haloperidol","ciprofloxacin","levofloxacin","clarithromycin","erythromycin","citalopram","escitalopram"],
     "Potential additive QT prolongation; assess ECG/QT and risk factors.")
]

def _norm(s: Optional[str]) -> str:
    return (s or "").strip().casefold()

def _hit(item_text: str, keys: List[str]) -> bool:
    t = item_text
    for k in keys:
        if k in t:
            return True
    return False

def _med_text(m: Dict[str, Any]) -> str:
    parts = [m.get("name") or m.get("substance") or "", m.get("strength") or "", m.get("dose") or ""]
    return " ".join([p for p in parts if p]).strip()

def check_allergy_cross_reactivity(allergies: List[str], meds: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    warnings = []
    alg_norm = [_norm(a) for a in allergies]
    # Map allergies to groups
    has_group = {g: any(any(alias in a for alias in v["aliases"]) for a in alg_norm) for g,v in ALLERGY_GROUPS.items()}
    for m in meds:
        text = _norm(_med_text(m))
        for (ag, sg, note) in CROSS_REACTIVITY_RULES:
            if has_group.get(ag, False):
                # Does med fall in suspect group?
                suspects = ALLERGY_GROUPS.get(sg, {}).get("aliases", [])
                if _hit(text, [s.casefold() for s in suspects]):
                    warnings.append({"type":"allergy_crossreactivity", "against": ag, "med": m, "note": note})
    return warnings

def check_interactions(meds: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    # Create flat list of med strings to match
    texts = [_norm(_med_text(m)) for m in meds]
    pair_warnings = []
    n = len(meds)
    for i in range(n):
        for j in range(i+1, n):
            t1, t2 = texts[i], texts[j]
            for A, B, note in INTERACTION_RULES:
                if (_hit(t1, [a.casefold() for a in A]) and _hit(t2, [b.casefold() for b in B])) or \
                   (_hit(t2, [a.casefold() for a in A]) and _hit(t1, [b.casefold() for b in B])):
                    pair_warnings.append({"type":"interaction", "med1": meds[i], "med2": meds[j], "note": note})
    # Deduplicate warnings by note + meds
    uniq = []
    seen = set()
    for w in pair_warnings:
        key = (w["note"], _med_text(w["med1"]), _med_text(w["med2"]))
        if key not in seen:
            uniq.append(w); seen.add(key)
    return uniq

# UI
if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W, pandas as pd
        from IPython.display import display, HTML

        img_path = W.Text(description="Plan image", placeholder="/path/to/photo_or_scan.png")
        paste = W.Textarea(description="Paste payload", placeholder="(Optional) Paste decoded XML or BK/UKF text here…", layout=W.Layout(width="100%", height="120px"))
        allergies_in = W.Text(description="Allergies", placeholder="e.g. penicillin; ibuprofen", layout=W.Layout(width="60%"))
        run = W.Button(description="Scan/Parse", button_style="primary")
        out = W.Output()

        def on_run(_):
            with out:
                out.clear_output()
                payload = None
                if img_path.value:
                    payload = _decode_datamatrix_from_image(img_path.value)
                    if payload:
                        print("Decoded DataMatrix payload (first 200 chars):", payload[:200], "…")
                if not payload and paste.value.strip():
                    payload = paste.value.strip()
                    print("Using pasted payload.")
                if not payload:
                    print("No payload found. Install pylibdmtx or paste XML from the 2D code.")
                    return
                # If payload looks like XML (<MP ...>), parse; else just show raw
                if "<" in payload and ">" in payload:
                    res = _parse_bmp_xml(payload)
                    if not res.get("ok"):
                        print("Parse error:", res.get("error"))
                        return
                    meds = res["meds"]
                    if not meds:
                        print("No medication entries detected in XML. Showing raw payload snippet:\n", payload[:400], "…")
                        return
                    df = pd.DataFrame([{
                        "PZN": m.get("pzn"),
                        "Name/Substance": m.get("name") or m.get("substance"),
                        "Strength": m.get("strength"),
                        "Dose": m.get("dose"),
                        "Route": m.get("route"),
                        "Form": m.get("form"),
                    } for m in meds])
                    display(HTML("<b>Parsed medications (from BMP)</b>"))
                    display(df.style.hide(axis='index'))

                    allergies = [a.strip() for a in allergies_in.value.split(";") if a.strip()]
                    alg_warn = check_allergy_cross_reactivity(allergies, meds) if allergies else []
                    int_warn = check_interactions(meds)

                    if alg_warn or int_warn:
                        print("\nWarnings (review; not therapeutic decisions):")
                        for w in alg_warn:
                            print(f" - Allergy cross-reactivity ({w['against']}): { _med_text(w['med']) } → {w['note']}")
                        for w in int_warn:
                            print(f" - Interaction: { _med_text(w['med1']) } + { _med_text(w['med2']) } → {w['note']}")
                    else:
                        print("No heuristic warnings found (rule-base is limited; always review clinically).")

                    # Stage import payload (FHIR-like MedicationStatement draft)
                    bundle = {
                        "resourceType": "Bundle",
                        "type": "collection",
                        "timestamp": datetime.now(timezone.utc).isoformat(),
                        "entry": []
                    }
                    for m in meds:
                        entry = {
                            "resource": {
                                "resourceType": "MedicationStatement",
                                "status": "active",
                                "medicationCodeableConcept": {
                                    "text": m.get("name") or m.get("substance") or (m.get("pzn") and f"PZN {m['pzn']}") or "Unknown"
                                },
                                "dosage": [{
                                    "text": " ".join([x for x in [m.get("dose"), m.get("freq"), m.get("route")] if x])
                                }]
                            }
                        }
                        # Attach PZN as identifier if we have it
                        if m.get("pzn"):
                            entry["resource"]["medicationCodeableConcept"]["coding"] = [{
                                "system": "https://fhir.kbv.de/CodeSystem/KBV_CS_VS_PZN",
                                "code": m["pzn"]
                            }]
                        bundle["entry"].append(entry)
                    # Write draft bundle to /mnt/data for later Orbis integration
                    out_path = Path("/mnt/data/bmp_import_bundle.json")
                    out_path.write_text(json.dumps(bundle, ensure_ascii=False, indent=2))
                    print("\nDraft import bundle written to:", str(out_path))

                else:
                    print("Payload does not look like XML. Showing first 400 chars:\n", payload[:400], "…")
        run.on_click(on_run)

        display(W.VBox([
            W.HTML("<b>Bundeseinheitlicher Medikationsplan (BMP) → decode → parse → warn → stage import</b>"),
            img_path, paste, allergies_in, run, out
        ]))

    except Exception as e:
        print("BMP scanner UI unavailable:", e)
else:
    print("BMP scanner deferred… set CONFIG['RUN_UI']=True.")


BMP scanner deferred… set CONFIG['RUN_UI']=True.


In [47]:

# === Medication Plan Import + Checks (inline demo) ===
from __future__ import annotations
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime, timezone
import re, json

EVENT_LOG = Path(CONFIG["EVENT_LOG_PATH"])

def _append_event(ev: Dict[str, Any]):
    EVENT_LOG.parent.mkdir(parents=True, exist_ok=True)
    EVENT_LOG.touch(exist_ok=True)
    ev = {"ts": datetime.now(timezone.utc).isoformat(), **ev}
    with EVENT_LOG.open("a") as fp:
        fp.write(json.dumps(ev, ensure_ascii=False) + "\n")

# Minimal demo ATC dictionary (extend as needed)
ATC = {
    "ASS": "B01AC06", "Aspirin": "B01AC06", "Acetylsalicylsäure": "B01AC06",
    "Ibuprofen": "M01AE01",
    "Metformin": "A10BA02",
    "Ramipril": "C09AA05",
    "Simvastatin": "C10AA01",
    "Amoxicillin": "J01CA04", "Amoxicillin/Clavulansäure": "J01CR02",
    "Clarithromycin": "J01FA09",
    "Phenprocoumon": "B01AA04",
    "Spironolacton": "C03DA01",
}

# Demo interaction rules (replace with real rules JSON at MED_RULES_PATH for production)
DEMO_RULES = [
    # tuples of ATC or name patterns with a message
    (["J01FA09","C10AA01"], "Clarithromycin + Simvastatin: risk of myopathy/rhabdomyolysis"),
    (["C09AA05","C03DA01"], "ACE inhibitor + Spironolactone: risk of hyperkalaemia"),
    (["M01AE01","C09AA05"], "NSAID (Ibuprofen) + ACE inhibitor (Ramipril): risk of renal impairment"),
    (["B01AA04","B01AC06"], "Phenprocoumon + ASS: increased bleeding risk"),
]

# Very conservative allergy heuristics (demo)
ALLERGY_CLASS_MAP = {
    "penicillin": ["J01C", "J01CA", "J01CR", "J01CF"],
    "ibuprofen": ["M01AE"],
    "aspirin": ["B01AC06","N02BA"],
}

def _load_rules() -> List[Tuple[List[str], str]]:
    path = Path(CONFIG["MED_RULES_PATH"])
    if path.exists():
        try:
            js = json.loads(path.read_text())
            # expect [{"combo":["ATC1","ATC2"], "msg":"..."}]
            rules = []
            for r in js:
                combo = r.get("combo") or r.get("codes") or []
                msg = r.get("msg") or r.get("message") or "Interaction"
                if isinstance(combo, list) and combo:
                    rules.append((combo, msg))
            if rules:
                return rules
        except Exception:
            pass
    return DEMO_RULES

def _load_allergies() -> List[str]:
    path = Path(CONFIG["ALLERGIES_PATH"])
    if path.exists():
        try:
            js = json.loads(path.read_text())
            if isinstance(js, dict) and "allergies" in js and isinstance(js["allergies"], list):
                return js["allergies"]
        except Exception:
            pass
    return []

FREQ_MAP = {
    "morgens": "morning", "mittags": "noon", "abends": "evening", "nachts": "night",
    "1-0-0": "morning", "0-1-0": "noon", "0-0-1": "evening", "1-1-1": "tid",
    "1-0-1": "bid", "1-1-0": "bid", "2-0-0": "morning*2"
}

@dataclass
class MedEntry:
    name: str
    atc: Optional[str]
    strength: Optional[str]
    route: Optional[str]
    freq: Optional[str]
    prn: bool

def _normalize_name(name: str) -> str:
    name = name.strip()
    # strip common forms
    name = re.sub(r"\b(retard|sr|tbl|tab|tabl|tabletten|kapseln|lösung|saf t|sft)\b", "", name, flags=re.I)
    return re.sub(r"\s+", " ", name).strip()

def _guess_atc(name: str) -> Optional[str]:
    key = name
    if key in ATC: return ATC[key]
    # try title-case exact
    key = name.split()[0].capitalize()
    return ATC.get(key)

def parse_bmp_text(txt: str) -> List[MedEntry]:
    meds: List[MedEntry] = []
    # Split lines and parse patterns like "Metformin 1000 mg 1-0-1 p.o." or "Ibuprofen 400 mg bei Bedarf"
    for raw in txt.splitlines():
        line = raw.strip()
        if not line or line.startswith("#"): continue
        # Extract med name (words until first number or PRN marker)
        m = re.match(r"(?P<name>[A-Za-zÄÖÜäöüß/ \-]+)\s+(?P<rest>.*)", line)
        if not m: 
            continue
        name = _normalize_name(m.group("name"))
        rest = m.group("rest")

        # strength + unit
        m2 = re.search(r"(?P<dose>\d+(?:[.,]\d+)?)\s*(?P<unit>mg|g|ml|IE|µg)", rest, flags=re.I)
        strength = None
        if m2:
            strength = f"{m2.group('dose').replace(',', '.')} {m2.group('unit').upper()}"

        # route
        m3 = re.search(r"\b(p\.o\.|i\.v\.|s\.c\.|i\.m\.|p\.r\.)\b", rest, flags=re.I)
        route = m3.group(0).lower() if m3 else None

        # frequency
        m4 = re.search(r"\b(\d-\d-\d)\b", rest)
        freq = None
        if m4:
            freq = FREQ_MAP.get(m4.group(1), m4.group(1))
        else:
            # words
            for de, en in FREQ_MAP.items():
                if re.search(rf"\b{re}\b", rest, flags=re.I):
                    freq = en; break

        prn = bool(re.search(r"\b(prn|bedarf|bei bedarf)\b", rest, flags=re.I))
        atc = _guess_atc(name)

        meds.append(MedEntry(name=name, atc=atc, strength=strength, route=route, freq=freq, prn=prn))
    return meds

def check_interactions(meds: List[MedEntry]) -> List[Dict[str, Any]]:
    rules = _load_rules()
    codes = set([m.atc for m in meds if m.atc])
    names = set([m.name for m in meds])
    findings = []
    for combo, msg in rules:
        # treat combo entries as ATC prefixes or exact names/ATC
        present = 0
        for token in combo:
            token = str(token)
            if token in codes or token in names:
                present += 1
                continue
            # ATC prefix match (e.g., J01C* class)
            if any(c and c.startswith(token) for c in codes):
                present += 1
        if present == len(combo):
            findings.append({"type":"med_interaction_warning","msg": msg, "combo": combo})
    return findings

def check_allergies(meds: List[MedEntry], allergies: List[str]) -> List[Dict[str, Any]]:
    finds = []
    norm_all = [a.lower() for a in allergies]
    for m in meds:
        # direct string hit
        if any(a in m.name.lower() for a in norm_all):
            finds.append({"type":"med_allergy_warning","msg": f"Allergy matches {m.name}", "med": m.name})
        # class proximity
        if m.atc:
            for a in norm_all:
                classes = ALLERGY_CLASS_MAP.get(a)
                if not classes: continue
                if any(m.atc.startswith(pref) for pref in classes):
                    finds.append({"type":"med_allergy_warning","msg": f"Allergy '{a}' overlaps ATC class of {m.name} ({m.atc})", "med": m.name})
    return finds

def emit_med_events(patient_id: str, meds: List[MedEntry]):
    # summary
    _append_event({"type":"med_plan_import","patient_id": patient_id, "n": len(meds)})
    for m in meds:
        _append_event({"type":"med_entry","patient_id": patient_id, "name": m.name, "atc": m.atc,
                       "strength": m.strength, "route": m.route, "freq": m.freq, "prn": m.prn})

def process_bmp_text(patient_id: str, txt: str):
    meds = parse_bmp_text(txt)
    emit_med_events(patient_id, meds)
    warnings = []
    warnings += check_interactions(meds)
    warnings += check_allergies(meds, _load_allergies())
    for w in warnings:
        _append_event({**w, "patient_id": patient_id})
    return meds, warnings

DEMO_BMP = """\
# Demo Bundeseinheitlicher Medikationsplan (free-text)
Metformin 1000 mg 1-0-1 p.o.
Ramipril 5 mg 1-0-0 p.o.
Ibuprofen 400 mg bei Bedarf p.o.
Simvastatin 20 mg 0-0-1 p.o.
Clarithromycin 500 mg 1-0-1 p.o. (für 5 Tage)
Phenprocoumon 3 mg 1-0-0 p.o.
"""

if CONFIG.get("RUN_MEDS"):
    # For non-UI runs, process DEMO_BMP for patient SYN-DEMO-1
    meds, warns = process_bmp_text("SYN-DEMO-1", DEMO_BMP)
    print(f"[MEDS] Parsed {len(meds)} meds; warnings: {len(warns)}")
else:
    print("Deferred… set CONFIG['RUN_MEDS']=True for meds demo.")


Deferred… set CONFIG['RUN_MEDS']=True for meds demo.


In [48]:

# UI: Medication Plan Import (Paste or OCR), interaction + allergy checks
if CONFIG.get("RUN_UI") and CONFIG.get("RUN_MEDS"):
    try:
        import ipywidgets as W
        import pandas as pd
        import os
        
        pid = W.Text(description="Patient ID", value="SYN-DEMO-1")
        path = W.Text(description="Scan path", placeholder="/mnt/data/scan.pdf or .png")
        ocr_btn = W.Button(description="Run OCR")
        txt = W.Textarea(value=DEMO_BMP, description="Plan text", layout=W.Layout(width="100%", height="160px"))
        parse_btn = W.Button(description="Parse & Check", button_style="primary")
        out = W.Output()

        def _ocr_run(_):
            with out:
                out.clear_output()
                try:
                    p = Path(path.value.strip())
                    if not p.exists():
                        print("File not found:", p)
                        return
                    # Best-effort OCR: pdf2image + pytesseract if present
                    txt_val = None
                    try:
                        import pytesseract
                        from PIL import Image
                        if p.suffix.lower()==".pdf":
                            from pdf2image import convert_from_path
                            pages = convert_from_path(str(p))
                            text_parts = []
                            for page in pages:
                                text_parts.append(pytesseract.image_to_string(page, lang="deu"))
                            txt_val = "\n".join(text_parts)
                        else:
                            img = Image.open(str(p))
                            txt_val = pytesseract.image_to_string(img, lang="deu")
                    except Exception as e:
                        print("OCR unavailable or failed:", e)
                    if txt_val:
                        txt.value = txt_val
                        print("OCR complete → text area filled.")
                except Exception as e:
                    print("OCR error:", e)

        def _parse_run(_):
            with out:
                out.clear_output()
                meds, warns = process_bmp_text(pid.value.strip() or "DEMO", txt.value)
                if meds:
                    df = pd.DataFrame([m.__dict__ for m in meds])
                    display(df.style.hide(axis='index'))
                if warns:
                    print("\nWarnings:")
                    for w in warns:
                        print(" -", w["type"], ":", w.get("msg"))
                print("\nEvents appended to", CONFIG["EVENT_LOG_PATH"])

        ocr_btn.on_click(_ocr_run); parse_btn.on_click(_parse_run)
        display(W.VBox([W.HBox([pid]), W.HBox([path, ocr_btn]), txt, parse_btn, out]))
    except Exception as e:
        print("UI unavailable:", e)


In [49]:

# Conference Demo: one-click run (synthetic ML + meds import + ICU ETA)
if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W, json, pandas as pd, importlib.util
        run_btn = W.Button(description="Run Full Demo", button_style="success")
        out = W.Output()

        def _run(_):
            with out:
                out.clear_output()
                # 1) Synthetic ML pipeline (if configured)
                try:
                    CONFIG["RUN_SYNTH"] = True
                    print("[1/3] Running synthetic ML pipeline…")
                    # reuse the run_synth_pipeline if defined
                    try:
                        res = run_synth_pipeline(save_csv=False)
                    except NameError:
                        print("run_synth_pipeline not loaded in this kernel; skip")
                    else:
                        print("  →", res)
                except Exception as e:
                    print("Synthetic pipeline error:", e)

                # 2) ICU ETAs (if ICU panel code is present)
                try:
                    from datetime import datetime, timezone
                    # create a tiny scenario: set one unit to open in 30 min
                    state = {"timestamp": datetime.now(timezone.utc).isoformat(),
                             "units": [{"name":"1A Neurochirurgische Intensivstation","capacity":12,"occupied":12,"discharge_eta_minutes":[30,120]},
                                       {"name":"H2b Intensivstation Gefäß- und Herzmedizin","capacity":8,"occupied":7,"discharge_eta_minutes":[]} ]}
                    Path("/mnt/data/icu_status.json").write_text(json.dumps(state, ensure_ascii=False, indent=2))
                    print("[2/3] ICU sample state saved → /mnt/data/icu_status.json")
                except Exception as e:
                    print("ICU demo setup error:", e)

                # 3) Meds import demo
                try:
                    print("[3/3] Parsing demo BMP for patient SYN-DEMO-1…")
                    meds, warns = process_bmp_text("SYN-DEMO-1", DEMO_BMP)
                    print(f"  → {len(meds)} meds, {len(warns)} warnings; events appended.")
                except Exception as e:
                    print("Meds demo error:", e)

                # Tail event log
                try:
                    p = Path(CONFIG["EVENT_LOG_PATH"])
                    if p.exists():
                        lines = p.read_text().strip().splitlines()[-10:]
                        print("\nEvent log tail:")
                        for ln in lines:
                            print(" ", ln[:200])
                except Exception as e:
                    print("Log tail error:", e)

        run_btn.on_click(_run)
        display(W.VBox([run_btn, out]))
    except Exception as e:
        print("Demo UI unavailable:", e)
else:
    print("Demo UI deferred… set CONFIG['RUN_UI']=True.")


Demo UI deferred… set CONFIG['RUN_UI']=True.


In [50]:

# === Medication Plan Import + Checks (inline, no sidecars) ===
from __future__ import annotations
from typing import Dict, Any, List, Optional, Tuple
from pathlib import Path
from datetime import datetime, timezone
import json, re

EVENT_LOG = Path(CONFIG["EVENT_LOG_PATH"])

def _append_event(ev: Dict[str, Any]):
    EVENT_LOG.parent.mkdir(parents=True, exist_ok=True)
    EVENT_LOG.touch(exist_ok=True)
    ev = {"ts": datetime.now(timezone.utc).isoformat(), **ev}
    with EVENT_LOG.open("a") as fp:
        fp.write(json.dumps(ev, ensure_ascii=False) + "\n")

# Minimal ATC/name map for demo (extendable)
_ATC_MAP = {
    "amoxicillin": {"atc": "J01CA04", "class": "penicillin"},
    "ibuprofen": {"atc": "M01AE01", "class": "nsaid"},
    "warfarin": {"atc": "B01AA03", "class": "coumarin"},
    "ramipril": {"atc": "C09AA05", "class": "ace"},
    "spironolacton": {"atc": "C03DA01", "class": "aldosterone_antagonist"},
    "metoprolol": {"atc": "C07AB02", "class": "beta_blocker"},
    "simvastatin": {"atc": "C10AA01", "class": "statin"},
    "clarithromycin": {"atc": "J01FA09", "class": "macrolide"},
    "azithromycin": {"atc": "J01FA10", "class": "macrolide"},
}

# Demo interaction rules (replace via MED_RULES_PATH for your own rules)
_DEMO_RULES = [
    {"a": "ibuprofen", "b": "warfarin", "severity": "major", "message": "Bleeding risk (NSAID + warfarin)"},
    {"a": "ramipril", "b": "spironolacton", "severity": "moderate", "message": "Hyperkalemia risk (ACE + aldosterone antagonist)"},
    {"a": "amoxicillin", "b": "warfarin", "severity": "moderate", "message": "Antibiotic may potentiate warfarin effect"},
    {"a": "clarithromycin", "b": "simvastatin", "severity": "major", "message": "Rhabdomyolysis risk (CYP3A4 inhibition)"},
]

_ROUTES = ["p.o.", "po", "i.v.", "iv", "i.m.", "im", "s.c.", "sc", "inhalativ", "topisch", "nasal", "otic", "ophthalmic"]
_FREQ_WORDS = ["morgens", "mittags", "abends", "nachts"]
_FREQ_PAT = re.compile(r"\b(\d+)[-/.](\d+)[-/.](\d+)\b")
_DOSE_PAT = re.compile(r"(\d+(?:[.,]\d+)?)\s*(mg|g|mcg|µg|ml|IE|Einheiten)\b", flags=re.I)

def _normalize_name(s: str) -> str:
    s = s.strip().lower()
    s = s.replace("ä","ae").replace("ö","oe").replace("ü","ue").replace("ß","ss")
    return re.sub(r"[^a-z0-9]+", " ", s).strip()

def _map_to_atc(name_norm: str) -> Dict[str, Any]:
    for key, meta in _ATC_MAP.items():
        if key in name_norm:
            return {"name_norm": key, **meta}
    return {"name_norm": name_norm, "atc": None, "class": None}

def parse_med_line(line: str) -> Optional[Dict[str, Any]]:
    raw = line.strip()
    if not raw or raw.startswith("#"):
        return None
    name = raw.split(",")[0].split("  ")[0]  # up to comma or double spaces
    name_norm = _normalize_name(name)
    dose_val, dose_unit = None, None
    m = _DOSE_PAT.search(raw)
    if m:
        dose_val = float(m.group(1).replace(",", "."))
        dose_unit = m.group(2).lower()
    freq = None
    m = _FREQ_PAT.search(raw)
    if m:
        freq = f"{m.group(1)}-{m.group(2)}-{m.group(3)}"
    else:
        words = [w for w in _FREQ_WORDS if w in raw.lower()]
        if words:
            freq = ",".join(words)
    route = None
    for r in _ROUTES:
        if r in raw.lower():
            route = r
            break
    prn = bool(re.search(r"\b(prn|bei bedarf)\b", raw, flags=re.I))
    atc_meta = _map_to_atc(name_norm)
    return {
        "raw": raw,
        "name": name.strip(),
        "name_norm": atc_meta["name_norm"],
        "atc": atc_meta["atc"],
        "class": atc_meta["class"],
        "dose_value": dose_val,
        "dose_unit": dose_unit,
        "frequency": freq,
        "route": route,
        "prn": prn,
    }

def parse_med_text(text: str) -> List[Dict[str, Any]]:
    meds = []
    for line in text.splitlines():
        rec = parse_med_line(line)
        if rec:
            meds.append(rec)
    return meds

def load_rules(path: str) -> List[Dict[str, Any]]:
    p = Path(path)
    if p.exists():
        try:
            js = json.loads(p.read_text())
            if isinstance(js, list):
                return js
        except Exception:
            pass
    return list(_DEMO_RULES)

def load_allergies(path: str) -> List[str]:
    p = Path(path)
    if p.exists():
        try:
            js = json.loads(p.read_text())
            if isinstance(js, dict) and "allergies" in js and isinstance(js["allergies"], list):
                return [str(x) for x in js["allergies"]]
        except Exception:
            pass
    return []

def check_interactions(meds: List[Dict[str, Any]], rules: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    out = []
    names = [m["name_norm"] for m in meds]
    for r in rules:
        a, b = r["a"], r["b"]
        if (a in names and b in names) or (b in names and a in names):
            out.append({"type": "med_interaction_warning", **r})
    return out

def check_allergies(meds: List[Dict[str, Any]], allergy_terms: List[str]) -> List[Dict[str, Any]]:
    out = []
    terms = [t.lower() for t in allergy_terms]
    for m in meds:
        # direct name match
        if any(t in m["name"].lower() for t in terms):
            out.append({"type": "med_allergy_warning", "med": m["name"], "match": "name"})
            continue
        # conservative penicillin class heuristic
        if any("penicillin" in t for t in terms):
            if m["class"] == "penicillin" or m["name"].lower().endswith("cillin"):
                out.append({"type": "med_allergy_warning", "med": m["name"], "match": "class_penicillin"})
    return out

def ocr_file(path: str) -> Optional[str]:
    p = Path(path)
    if not p.exists():
        return None
    try:
        from PIL import Image
        import pytesseract
        if p.suffix.lower() in [".png",".jpg",".jpeg",".tif",".tiff"]:
            return pytesseract.image_to_string(Image.open(p))
        if p.suffix.lower() == ".pdf":
            try:
                from pdf2image import convert_from_path
                pages = convert_from_path(str(p))
                text = ""
                for img in pages:
                    text += pytesseract.image_to_string(img) + "\n"
                return text
            except Exception:
                return None
    except Exception:
        return None
    return None

def emit_meds_events(patient_id: str, meds: List[Dict[str, Any]], warnings: List[Dict[str, Any]]):
    _append_event({"type": "med_plan_import", "patient_id": patient_id, "n_meds": len(meds)})
    for m in meds:
        _append_event({"type": "med_entry", "patient_id": patient_id, **m})
    for w in warnings:
        _append_event({**w, "patient_id": patient_id})

if CONFIG.get("RUN_MEDS"):
    print("Medication import/checks enabled. Use the UI panel (if RUN_UI=True) or call the functions above.")
else:
    print("Deferred… set CONFIG['RUN_MEDS']=True to enable medication import & checks.")

# Optional UI
if CONFIG.get("RUN_UI") and CONFIG.get("RUN_MEDS"):
    try:
        import ipywidgets as W
        import pandas as pd
        pid = W.Text(description="Patient ID", placeholder="e.g., UKE-12345")
        path = W.Text(description="Scan path", placeholder="/mnt/data/scan.pdf (optional)")
        ocr_btn = W.Button(description="Run OCR")
        ta = W.Textarea(description="Plan text", layout=W.Layout(width="100%", height="180px"))
        parse_btn = W.Button(description="Parse & Check", button_style="primary")
        out = W.Output()

        def on_ocr(_):
            with out:
                out.clear_output()
                txt = ocr_file(path.value.strip())
                if txt:
                    ta.value = txt
                    print("OCR complete.")
                else:
                    print("OCR unavailable or failed. Paste the text instead.")

        def on_parse(_):
            with out:
                out.clear_output()
                text = ta.value.strip()
                if not text:
                    print("No text provided."); return
                meds = parse_med_text(text)
                rules = load_rules(CONFIG["MED_RULES_PATH"])
                allergies = load_allergies(CONFIG["ALLERGIES_PATH"])
                warnings = check_interactions(meds, rules) + check_allergies(meds, allergies)
                emit_meds_events(pid.value or "DEMO", meds, warnings)
                if meds:
                    display(pd.DataFrame(meds))
                else:
                    print("No medications parsed.")
                if warnings:
                    print("\nWarnings:")
                    display(pd.DataFrame(warnings))
                else:
                    print("\nNo interaction/allergy warnings.")

        ocr_btn.on_click(on_ocr)
        parse_btn.on_click(on_parse)
        display(W.VBox([pid, path, W.HBox([ocr_btn, parse_btn]), ta, out]))
    except Exception as e:
        print("Meds UI unavailable:", e)


Deferred… set CONFIG['RUN_MEDS']=True to enable medication import & checks.


In [51]:

# === Conference Demo: one-click walkthrough ===
from pathlib import Path
from datetime import datetime, timezone
import json

def _tail_events(n=12):
    p = Path(CONFIG["EVENT_LOG_PATH"])
    if not p.exists():
        return []
    lines = p.read_text().splitlines()
    return [json.loads(x) for x in lines[-n:]] if lines else []

if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W, pandas as pd

        btn = W.Button(description="Conference Demo → Run Full Demo", button_style="success")
        out = W.Output()

        def run_demo(_):
            with out:
                out.clear_output()
                print("Running conference demo…")

                # Step 1: Synthetic ML pipeline (if enabled)
                if CONFIG.get("RUN_SYNTH"):
                    try:
                        res = run_synth_pipeline(save_csv=False)
                        print("[SYNTH]", res)
                    except Exception as e:
                        print("[SYNTH ERROR]", e)
                else:
                    print("[SYNTH] Skipped (set RUN_SYNTH=True to include)")

                # Step 2: ICU snapshot → next bed in 30 min on one unit
                icu = {
                    "timestamp": datetime.now(timezone.utc).isoformat(),
                    "units": [
                        {"name":"1C Interdisziplinäre Intensivstation","capacity":12,"occupied":12,"discharge_eta_minutes":[30,120,240]},
                        {"name":"1G Internistische Intensivstation","capacity":12,"occupied":11,"discharge_eta_minutes":[60]}
                    ]
                }
                Path("/mnt/data/icu_status.json").write_text(json.dumps(icu, ensure_ascii=False, indent=2))
                print("[ICU] Snapshot saved → next bed on 1C in 30 min")

                # Step 3: Medication plan demo (paste-mode)
                demo_text = """\
Amoxicillin 500 mg 1-0-1 p.o.
Ibuprofen 400 mg 1-1-1 p.o. bei Bedarf
Warfarin 5 mg 1-0-0 p.o.
Ramipril 5 mg 1-0-0 p.o.
Spironolacton 25 mg 0-0-1 p.o.
"""
                meds = parse_med_text(demo_text)
                rules = load_rules(CONFIG["MED_RULES_PATH"])
                allergies = ["Penicillin"]  # demo allergy to trigger warning with Amoxicillin
                warnings = check_interactions(meds, rules) + check_allergies(meds, allergies)
                emit_meds_events("DEMO-123", meds, warnings)
                print("[MEDS] Parsed", len(meds), "entries; warnings:", len(warnings))

                # Show event log tail
                tail = _tail_events(12)
                if tail:
                    display(pd.DataFrame(tail))
                else:
                    print("No events yet.")

        btn.on_click(run_demo)
        display(W.VBox([btn, out]))
    except Exception as e:
        print("Demo UI unavailable:", e)
else:
    print("Demo UI deferred… set CONFIG['RUN_UI']=True.")


Demo UI deferred… set CONFIG['RUN_UI']=True.
